# SEKANS SAYISI DENEYİ — bağımsız notebook

Bu notebook, `ZoomInOut_v5i_02_dataset.ipynb`'in tamamını çalıştırmadan
**yalnızca sekans sayısı deneyini** yapmak için hazırlanmıştır. Havuz, bölme,
eşikler ve değerlendirme kümeleri ana koşuyla birebir aynıdır; tek değişken
üretilen sekans sayısıdır (600 → 800).

**GPU gerektirmeyen kısım:** Bölüm 1–6 ve sekans üretimi (Bölüm 7) tamamen
CPU'da çalışır. Tüm embedding ve CV önbellekleri Drive'da hazır olduğu için
model yüklenmez. GPU yalnızca Bölüm 9'daki 5-seed eğitim için gerekir.

**Kesintiye dayanıklıdır:** Üretim her 50 sekansta bir ilerlemeyi kaydeder;
kesilirse aynı hücre kaldığı yerden devam eder.

**Hiçbir mevcut sonucun üzerine yazmaz:** çıktılar `v5i_02_ps_seq800_*`
adlarıyla ayrı dosyalara gider.

## 1. Kurulum, veri sınıfları ve indeks

In [ ]:
# HÜCRE 1: Kurulum ve içe aktarımlar
!pip install -q datasets opencv-python Pillow tqdm nltk scikit-image

import os
import re
import copy
import pickle
import random
import secrets
import itertools
import textwrap
from dataclasses import dataclass, field
from typing import List, Tuple, Optional, Set

import time
import cv2
import nltk
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from datasets import load_from_disk
from tqdm.auto import tqdm

from google.colab import drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

try:
    nltk.data.find('tokenizers/punkt_tab')
except LookupError:
    nltk.download('punkt_tab', quiet=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Hazır! Aktif cihaz: {device}")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DRIVE_BASE = "/content/drive/MyDrive/COSMOS"

# --- v5'e ÖZEL dosya adları -- v4/Fikir2 ile ÇAKIŞMAZ, paralel çalıştırılabilir.
# CV_CACHE_PATH kasıtlı olarak YENİ bir isim -- ozellikler zenginlestirildigi
# icin v4'un eski onbellegini KULLANMIYORUZ (yoksa eski, zayif ozellikler
# sessizce geri yuklenir).
TRAIN_PAIRS_PATH   = f"{DRIVE_BASE}/ZoomInOut/v5i_02_train_pairs.pkl"  # v5i_02: genisletilmis (yeni COCO gorselli) havuz
TEST_PAIRS_PATH    = f"{DRIVE_BASE}/ZoomInOut/v5i_02_test_pairs.pkl"  # v5i_02
VAL_PAIRS_PATH     = f"{DRIVE_BASE}/ZoomInOut/v5i_02_val_pairs.pkl"  # v5i_02
CHAIN_RECORDS_PATH = f"{DRIVE_BASE}/ZoomInOut/v5i_02_chain_records.pkl"  # v5i_02
CV_CACHE_PATH      = f"{DRIVE_BASE}/ZoomInOut/v5_cv_feature_cache_enriched.pkl"
MODE_CALIB_PATH    = f"{DRIVE_BASE}/ZoomInOut/v5_mode_calibration_percentile.pkl"  # v5b: eski (mean,std) onbellegiyle CAKISMASIN diye yeni isim
COCO_CAPTIONS_EMB_PATH = f"{DRIVE_BASE}/ZoomInOut/v5_coco_captions_embedded.pkl"


In [ ]:
# HÜCRE 2: Dataclass'lar

@dataclass
class ImagePatchMeta:
    image_id:         str
    patch_id:         str
    bbox:             list
    embedding:        np.ndarray
    objectness_score: float = 1.0
    type:             str   = "object"
    window_type:      str   = ""
    area:             float = 0.0
    attention_score:  float = 0.0
    color_feat:       Optional[np.ndarray] = None
    shape_feat:       Optional[np.ndarray] = None
    texture_feat:     Optional[np.ndarray] = None

    def iou(self, other: "ImagePatchMeta") -> float:
        x1 = max(self.bbox[0], other.bbox[0])
        y1 = max(self.bbox[1], other.bbox[1])
        x2 = min(self.bbox[2], other.bbox[2])
        y2 = min(self.bbox[3], other.bbox[3])
        inter = max(0, x2 - x1) * max(0, y2 - y1)
        if inter == 0:
            return 0.0
        a1 = (self.bbox[2] - self.bbox[0]) * (self.bbox[3] - self.bbox[1])
        a2 = (other.bbox[2] - other.bbox[0]) * (other.bbox[3] - other.bbox[1])
        return inter / (a1 + a2 - inter + 1e-10)


@dataclass
class TextPatchMeta:
    doc_id:         str
    patch_id:       str
    text_content:   str
    embedding:      np.ndarray
    win_size:       int   = 2
    win_label:      str   = "w2"
    richness_score: float = 0.0
    sent_range:     tuple = field(default_factory=lambda: (0, 0))
    lexical_feat:   Optional[np.ndarray] = None  # TF-IDF -- metin-tarafı hard-negative madenciliği için


@dataclass
class HybridStepRecord:
    step:            int
    action:          str
    modality:        str
    image_id:        Optional[str]   = None
    patch_id:        Optional[str]   = None
    bbox:            Optional[list]  = None
    doc_id:          Optional[str]   = None
    text_content:    Optional[str]   = None
    similarity:      float           = 0.0
    similarity_mode: str             = "semantic"
    verified_label:  str             = "positive"
    # Metin-tarafı hard-negative (sadece img_to_txt/txt_to_img adımlarında dolu olur):
    hard_negative_txt_patch_id: Optional[str]   = None
    hard_negative_txt_sim:      Optional[float] = None
    hard_negative_txt_mode:     Optional[str]   = None


In [ ]:
# HÜCRE 3: HybridMultimodalIndex

class HybridMultimodalIndex:
    def __init__(self):
        self.image_patches: List[ImagePatchMeta] = []
        self.text_patches:  List[TextPatchMeta]  = []
        self._img_matrix:   Optional[np.ndarray] = None
        self._txt_matrix:   Optional[np.ndarray] = None
        self._lex_matrix:   Optional[np.ndarray] = None
        self._img_dirty = True
        self._txt_dirty = True
        self._lex_dirty = True

    def _is_quality_text(self, text: str) -> bool:
        if not text or len(text.split()) < 5:
            return False
        if text.strip().endswith("?"):
            return False
        BAGLAC = {"ve", "ama", "fakat", "lakin", "çünkü", "ancak",
                  "ile", "ya", "da", "de", "ki", "bu", "şu", "o",
                  "diye", "hem", "ne", "ah", "oh", "hmm"}
        ilk = text.split()[0].lower().rstrip(".,!?")
        if ilk in BAGLAC:
            return False
        if text.startswith('"') or text.startswith('-') or '—' in text:
            return False
        return True

    def add_image_patch(self, patch: ImagePatchMeta):
        for attr in ("embedding", "color_feat", "shape_feat", "texture_feat"):
            feat = getattr(patch, attr, None)
            if feat is not None:
                n = np.linalg.norm(feat)
                if n > 0:
                    setattr(patch, attr, feat / n)
        self.image_patches.append(patch)
        self._img_dirty = True

    def add_text_patch(self, patch: TextPatchMeta):
        n = np.linalg.norm(patch.embedding)
        if n > 0:
            patch.embedding = patch.embedding / n
        if patch.lexical_feat is not None:
            ln = np.linalg.norm(patch.lexical_feat)
            if ln > 0:
                patch.lexical_feat = patch.lexical_feat / ln
        self.text_patches.append(patch)
        self._txt_dirty = True
        self._lex_dirty = True

    def _build_matrices(self):
        if self._img_dirty and self.image_patches:
            self._img_matrix = np.stack([p.embedding for p in self.image_patches], axis=0).astype(np.float32)
            self._img_dirty = False
        if self._txt_dirty and self.text_patches:
            self._txt_matrix = np.stack([p.embedding for p in self.text_patches], axis=0).astype(np.float32)
            self._txt_dirty = False
        if self._lex_dirty and self.text_patches and all(p.lexical_feat is not None for p in self.text_patches):
            self._lex_matrix = np.stack([p.lexical_feat for p in self.text_patches], axis=0).astype(np.float32)
            self._lex_dirty = False

    def query_text(self, query_emb, exclude_doc_ids=None, threshold=-1.0, min_win_size=1, top_k=1):
        """v5i: top_k>1 verilirse, tek-en-iyi yerine EN IYI top_k adayi
        (agirlikli skora gore sirali) dondurur -- motor bunlar arasindan
        cesitlilik-farkinda secim yapabilsin diye (bkz. MultimodalSpiralEngine)."""
        self._build_matrices()
        if self._txt_matrix is None or not self.text_patches:
            return []
        exclude_doc_ids = exclude_doc_ids or set()
        q = query_emb / (np.linalg.norm(query_emb) + 1e-9)
        raw = np.clip(self._txt_matrix @ q, -1.0, 1.0)
        WIN_WEIGHT = {1: 0.70, 2: 1.05, 3: 1.15}
        weighted = raw.copy()
        for i, p in enumerate(self.text_patches):
            if p.doc_id in exclude_doc_ids or p.win_size < min_win_size or not self._is_quality_text(p.text_content):
                weighted[i] = -999.0
            else:
                weighted[i] = weighted[i] * WIN_WEIGHT.get(p.win_size, 1.0)
        if top_k <= 1:
            best_idx = int(np.argmax(weighted))
            if weighted[best_idx] <= -999.0:
                return []
            return [(self.text_patches[best_idx], float(raw[best_idx]))]
        order = np.argsort(-weighted)[:top_k]
        return [(self.text_patches[i], float(raw[i])) for i in order if weighted[i] > -999.0]

    def query_image(self, query_emb, exclude_image_ids=None, top_k=1):
        """v5i: top_k>1 verilirse, tek-en-iyi yerine EN IYI top_k adayi dondurur."""
        self._build_matrices()
        if self._img_matrix is None or not self.image_patches:
            return []
        exclude_image_ids = exclude_image_ids or set()
        q = query_emb / (np.linalg.norm(query_emb) + 1e-9)
        raw = np.clip(self._img_matrix @ q, -1.0, 1.0)
        TYPE_WEIGHT = {"object": 1.10, "background": 0.92}
        weighted = raw.copy()
        for i, p in enumerate(self.image_patches):
            if p.image_id in exclude_image_ids:
                weighted[i] = -999.0
            else:
                weighted[i] = weighted[i] * TYPE_WEIGHT.get(p.type, 1.0)
        if top_k <= 1:
            best_idx = int(np.argmax(weighted))
            if weighted[best_idx] <= -999.0:
                return []
            return [(self.image_patches[best_idx], float(raw[best_idx]))]
        order = np.argsort(-weighted)[:top_k]
        return [(self.image_patches[i], float(raw[i])) for i in order if weighted[i] > -999.0]

    def text_patches_of(self, doc_id):
        return [p for p in self.text_patches if p.doc_id == doc_id]

    def image_patches_of(self, image_id):
        return [p for p in self.image_patches if p.image_id == image_id]

    def image_ids(self):
        # v5h: SIRALI donduruluyor -- set() sirasi Python surecleri arasinda
        # hash-randomizasyonu yuzunden DEGISEBILIR, bu da train/test
        # bolunmesinin oturumdan oturuma FARKLI cikmasina (ve sizintiya)
        # yol aciyordu.
        return sorted({p.image_id for p in self.image_patches})

    def find_hard_negative_text(self, anchor_patch: "TextPatchMeta", exclude_doc_ids,
                                 hardness_min: float = 0.05, hard_neg_sem_thresh: float = 0.85):
        """Fikir 2: metin-tarafı hard-negative arama. anchor_patch ile YÜZEYDE
        (TF-IDF/leksik) benzer ama SEMANTİK olarak farklı, BAŞKA bir dokümandan
        bir metin adayı arar -- görsel tarafındaki renk/şekil/doku mantığının
        metin karşılığı. Traversal'ı ETKİLEMEZ, sadece paralel bir alan olarak
        (img_to_txt adımına) eklenir; txt_shift'in (chain tutarlılığı için
        kritik) semantik-en-yakın seçim mantığına dokunmaz.
        """
        self._build_matrices()
        if self._lex_matrix is None or anchor_patch.lexical_feat is None:
            return None
        surface_all = np.clip(self._lex_matrix @ anchor_patch.lexical_feat, -1.0, 1.0)
        sem_all = np.clip(self._txt_matrix @ anchor_patch.embedding, -1.0, 1.0)
        hardness_all = surface_all - sem_all

        best_idx, best_hardness = -1, -999.0
        for i, p in enumerate(self.text_patches):
            if p.doc_id in exclude_doc_ids or p.doc_id == anchor_patch.doc_id:
                continue
            if not self._is_quality_text(p.text_content):
                continue
            if hardness_all[i] > best_hardness:
                best_idx, best_hardness = i, hardness_all[i]

        if best_idx == -1 or best_hardness < hardness_min or sem_all[best_idx] >= hard_neg_sem_thresh:
            return None
        cand = self.text_patches[best_idx]
        return cand.patch_id, float(surface_all[best_idx]), "lexical"


In [ ]:
# HÜCRE 5a: Ham veri yükleme + indeks kurulumu (CV özellikleri HENÜZ yok)

hybrid_index = HybridMultimodalIndex()

text_dataset_path  = f'{DRIVE_BASE}/ds_sampled_text_datas_tr'
image_dataset_path = f'{DRIVE_BASE}/ds_sampled_image_datas_WH_224'

print("⏳ 1/4: Görsel veri kümesi yükleniyor...")
ds_sampled_image = load_from_disk(image_dataset_path)
print("⏳ 2/4: Metin veri kümesi yükleniyor...")
ds_sampled_text = load_from_disk(text_dataset_path)

print("⏳ 3/4: Metin embedding'leri indekse ekleniyor...")
with open(f'{DRIVE_BASE}/ZoomInOut/simple_story_sentences_embeddings_new_tr_5000.pkl', 'rb') as f:
    all_text_metadata = pickle.load(f)

from sklearn.feature_extraction.text import TfidfVectorizer
_tfidf = TfidfVectorizer(max_features=512, min_df=2)
_lex_matrix = _tfidf.fit_transform([m["text_content"] for m in all_text_metadata]).toarray().astype(np.float32)

for i, txt_m in enumerate(all_text_metadata):
    hybrid_index.add_text_patch(TextPatchMeta(
        doc_id=txt_m["doc_id"], patch_id=txt_m["patch_id"],
        text_content=txt_m["text_content"], embedding=txt_m["embedding"],
        win_size=int(txt_m.get("win_size", 2)), win_label=txt_m.get("win_label", "w2"),
        richness_score=float(txt_m.get("richness_score", 0.0)),
        lexical_feat=_lex_matrix[i],
    ))

print("⏳ 4/4: Görsel embedding'leri indekse ekleniyor...")
with open(f'{DRIVE_BASE}/ZoomInOut/coco_image_dynamic_embeddings_new_WH_224_2048.pkl', 'rb') as f:
    all_image_metadata = pickle.load(f)
for img_m in all_image_metadata:
    hybrid_index.add_image_patch(ImagePatchMeta(
        image_id=img_m["image_id"], patch_id=img_m["patch_id"], bbox=img_m["bbox"],
        embedding=img_m["embedding"], objectness_score=img_m.get("objectness_score", 1.0),
        type=img_m.get("type", "object"),
    ))

print(f"✅ İndeks hazır: {len(hybrid_index.image_patches)} görsel yama, {len(hybrid_index.text_patches)} metin yaması.")


In [ ]:
# HÜCRE 5b: Zenginleştirilmiş CV özellik çıkarımı (önbellekli)
def visual_loader(image_id):
    idx = int(image_id.split("_")[-1])
    return ds_sampled_image[idx]["image"].convert("RGB")


from skimage.feature import local_binary_pattern

if os.path.exists(CV_CACHE_PATH):
    print(f"⏳ Önbelleklenmiş ZENGİNLEŞTİRİLMİŞ CV özellikleri bulundu, hesaplama ATLANIYOR: {CV_CACHE_PATH}")
    with open(CV_CACHE_PATH, "rb") as f:
        cv_cache = pickle.load(f)
    n_hit = 0
    for patch in hybrid_index.image_patches:
        feats = cv_cache.get(patch.patch_id)
        if feats is not None:
            patch.color_feat, patch.shape_feat, patch.texture_feat = feats
            n_hit += 1
        else:
            patch.color_feat = patch.embedding
            patch.shape_feat = patch.embedding
            patch.texture_feat = patch.embedding
    print(f"✅ {n_hit}/{len(hybrid_index.image_patches)} yama önbellekten dolduruldu.")
else:
    print("⏳ Önbellek yok -- tüm görsel yamalar için ZENGİNLEŞTİRİLMİŞ renk/şekil/doku özellikleri hesaplanıyor (uzun sürebilir)...")
    for patch in tqdm(hybrid_index.image_patches, desc="CV öznitelikleri (zenginleştirilmiş)"):
        try:
            pil_img = visual_loader(patch.image_id)
            img_bgr = cv2.cvtColor(np.array(pil_img.convert("RGB")), cv2.COLOR_RGB2BGR)
            H_img, W_img = img_bgr.shape[:2]
            x1 = int(np.clip(patch.bbox[0] * (W_img / 224.0), 0, W_img - 1))
            y1 = int(np.clip(patch.bbox[1] * (H_img / 224.0), 0, H_img - 1))
            x2 = int(np.clip(patch.bbox[2] * (W_img / 224.0), 1, W_img))
            y2 = int(np.clip(patch.bbox[3] * (H_img / 224.0), 1, H_img))
            if x2 <= x1: x2 = x1 + 1
            if y2 <= y1: y2 = y1 + 1
            crop = img_bgr[y1:y2, x1:x2]

            # --- RENK: değişmedi (HSV histogramı) ---
            hsv = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)
            hist_h = cv2.calcHist([hsv], [0], None, [16], [0, 180])
            hist_s = cv2.calcHist([hsv], [1], None, [8], [0, 256])
            hist_v = cv2.calcHist([hsv], [2], None, [8], [0, 256])
            color_f = np.concatenate([hist_h, hist_s, hist_v]).flatten()
            color_f /= (np.linalg.norm(color_f) + 1e-10)

            gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)

            # --- DOKU: Sobel histogram (eski) + LBP histogram (YENİ) ---
            sobelx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
            sobely = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
            mag = np.sqrt(sobelx**2 + sobely**2)
            hist_txt, _ = np.histogram(mag, bins=32, range=(0, 255))
            hist_txt = hist_txt.astype(np.float32)

            lbp = local_binary_pattern(gray, P=8, R=1, method="uniform")
            hist_lbp, _ = np.histogram(lbp, bins=10, range=(0, 10))
            hist_lbp = hist_lbp.astype(np.float32)

            texture_f = np.concatenate([hist_txt, hist_lbp])
            texture_f /= (np.linalg.norm(texture_f) + 1e-10)

            # --- ŞEKİL: kenar histogram (16->32 bin) + Hu momentleri (YENİ) + aspect ratio ---
            edges = cv2.Canny(gray, 50, 150)
            hist_edge, _ = np.histogram(edges, bins=32, range=(0, 255))
            moments = cv2.moments(edges)
            hu = cv2.HuMoments(moments).flatten()
            hu_log = -np.sign(hu) * np.log10(np.abs(hu) + 1e-10)
            hu_log = np.nan_to_num(hu_log, nan=0.0, posinf=0.0, neginf=0.0)
            aspect_ratio = float(x2 - x1) / float(y2 - y1)
            shape_f = np.concatenate([hist_edge.astype(np.float32), hu_log.astype(np.float32), [aspect_ratio]])
            shape_f /= (np.linalg.norm(shape_f) + 1e-10)

            patch.color_feat = color_f
            patch.shape_feat = shape_f
            patch.texture_feat = texture_f
        except Exception:
            patch.color_feat = patch.embedding
            patch.shape_feat = patch.embedding
            patch.texture_feat = patch.embedding

    cv_cache = {p.patch_id: (p.color_feat, p.shape_feat, p.texture_feat) for p in hybrid_index.image_patches}
    with open(CV_CACHE_PATH, "wb") as f:
        pickle.dump(cv_cache, f)
    print(f"✅ Hesaplandı ve önbelleğe kaydedildi -> {CV_CACHE_PATH} (bir sonraki oturumda ATLANACAK)")


## 2. Mod kalibrasyonu, motor ve eşikler

In [ ]:
def calibrate_mode_stats(index, n_samples=3000, seed=42):
    """v5b: z-skor YERİNE tam sıralı örneklem (percentile-rank için) döndürür.
    z-skor denendiğinde std çok küçük olan modlar (shape: 0.0093) ufacık
    sapmalarda bile dev z-skorları üretip mod yarışmasını TAMAMEN ele
    geçiriyordu (renk %100 kazandı, shape/texture hiç seçilmedi). Percentile
    yaklaşımı her modun kendi dağılım şeklinden/varyansından bağımsız,
    adil bir [0,1] karşılaştırma sağlıyor."""
    rng = random.Random(seed)
    feat_attr = {"color": "color_feat", "shape": "shape_feat", "texture": "texture_feat"}
    image_ids = index.image_ids()
    stats = {}
    for mode, attr in feat_attr.items():
        sims = []
        tries = 0
        while len(sims) < n_samples and tries < n_samples * 5:
            tries += 1
            img_id = rng.choice(image_ids)
            patches = index.image_patches_of(img_id)
            if len(patches) < 2:
                continue
            a, b = rng.sample(patches, 2)
            fa, fb = getattr(a, attr, None), getattr(b, attr, None)
            if fa is None or fb is None:
                continue
            na, nb = np.linalg.norm(fa), np.linalg.norm(fb)
            if na == 0 or nb == 0:
                continue
            sims.append(float(np.dot(fa, fb) / (na * nb)))
        stats[mode] = np.sort(np.asarray(sims, dtype=np.float32))
    return stats


if os.path.exists(MODE_CALIB_PATH):
    print(f"⏳ Kaydedilmiş mod kalibrasyonu bulundu, yükleniyor: {MODE_CALIB_PATH}")
    with open(MODE_CALIB_PATH, "rb") as f:
        MODE_CALIB = pickle.load(f)
else:
    print("⏳ Mod kalibrasyonu hesaplanıyor (renk/şekil/doku için 3000'er rastgele-çift örneklemi)...")
    MODE_CALIB = calibrate_mode_stats(hybrid_index, n_samples=3000)
    with open(MODE_CALIB_PATH, "wb") as f:
        pickle.dump(MODE_CALIB, f)

print("\nMod kalibrasyonu (aynı-görsel-içi rastgele çiftlerin surface_sim dağılımı):")
for mode, sorted_sims in MODE_CALIB.items():
    print(f"  {mode:<10}: ortalama={sorted_sims.mean():.4f}  std={sorted_sims.std():.4f}  "
          f"(N={len(sorted_sims)}, percentile-rank ile kullanılacak)")


In [ ]:
# HÜCRE 4: MultimodalSpiralEngine

class MultimodalSpiralEngine:
    def __init__(self, index: HybridMultimodalIndex, similarity_threshold: float = 0.55,
                 max_iou_shift: float = 0.1, verbose: bool = True, min_win_size: int = 2,
                 hardness_min: float = 0.05, hard_neg_sem_thresh: float = 0.85,
                 mode_calib: Optional[dict] = None,
                 allowed_image_ids: Optional[Set[str]] = None,
                 top_k: int = 1, diversity_alpha: float = 0.0):
        self.index = index
        self.threshold = similarity_threshold
        self.max_iou_shift = max_iou_shift
        self.verbose = verbose
        self.min_win_size = min_win_size
        self.hardness_min = hardness_min
        self.hard_neg_sem_thresh = hard_neg_sem_thresh
        self._mode_field = {"semantic": "embedding", "color": "color_feat", "shape": "shape_feat", "texture": "texture_feat"}
        # v5: modlar-arasi adil karsilastirma icin z-normalizasyon istatistikleri
        # (bkz. Bolum 3b -- shape_feat'in dogmus olmasi shape modunu yapay
        # olarak kayirmasin diye).
        self.mode_calib = mode_calib or {}  # v5b: bos birak -> _calibrated ham sim'i aynen dondurur (fallback)
        # v5d: TRAIN/TEST leakage duzeltmesi -- motor, txt_to_img adiminda
        # zincirin baslangicindan BAGIMSIZ olarak TUM index'ten en yakin
        # goruseli ariyordu (train/test ayrimini bilmiyordu). allowed_image_ids
        # verilirse, arama havuzu SADECE bu kumeye kisitlanir.
        self.allowed_image_ids = set(allowed_image_ids) if allowed_image_ids is not None else None
        if self.allowed_image_ids is not None:
            self._forbidden_image_ids = set(self.index.image_ids()) - self.allowed_image_ids
        else:
            self._forbidden_image_ids = set()
        # v5i: HUB-ETKISI DUZELTMESI -- query_text/query_image eskiden saf
        # argmax'ti (deterministik, top_k=1), yani birbirine yakin embedding'li
        # farkli sorgular hep AYNI hedefe yakinsiyordu, ve 600 sekans arasinda
        # HICBIR paylasilan hafiza olmadigi icin bu yakinsama sessizce
        # tekrarlaniyordu (train havuzunun sadece %43'u kullanilmisti).
        # Simdi: top_k aday arasindan, ASIRI KULLANILANLARI cezalandirarak
        # (kullanim_sayisi * diversity_alpha kadar skor dusurerek) seciyoruz --
        # bu sayac TUM run_hybrid_loop cagrilari boyunca (yani tum sekanslar
        # arasinda) KALICI, kendi kendini duzelten adaptif bir mekanizma.
        self.top_k = top_k
        self.diversity_alpha = diversity_alpha
        self.image_use_count = {}
        self.doc_use_count = {}

    def _get_feat(self, patch, mode: str):
        return getattr(patch, self._mode_field.get(mode, "embedding"), None)

    def _calibrated(self, raw_sim: float, mode: str) -> float:
        """v5b: percentile-rank normalizasyonu (z-skor DEGIL -- z-skor, std'si
        cok kucuk olan modlari (shape) yapay sekilde ezmisti/kazandirmisti).
        mode_calib[mode], calibrate_mode_stats'tan gelen SIRALI bir dizi;
        raw_sim'in bu dizideki yuzdelik konumunu [0,1] olarak dondururuz."""
        sorted_sims = self.mode_calib.get(mode)
        if sorted_sims is None or not hasattr(sorted_sims, "__len__") or len(sorted_sims) == 0:
            return raw_sim
        rank = np.searchsorted(sorted_sims, raw_sim) / len(sorted_sims)
        return float(rank)

    def _cosine_sim(self, a, b, fallback=0.0):
        if a is None or b is None:
            return fallback
        na, nb = np.linalg.norm(a), np.linalg.norm(b)
        if na == 0 or nb == 0:
            return fallback
        return float(np.dot(a, b) / (na * nb))

    def _pick_diverse(self, candidates, use_count, key_fn):
        """v5i: top-K adaylar arasindan, kullanim sayisina gore cezalandirilmis
        skora bakarak sec (asiri kullanilan -- hub -- adaylar geri plana
        duser). Secilenin kullanim sayacini 1 artirir. Dondurulen skor HAM
        (cezasiz) skordur -- loglama/rapor tutarliligi icin."""
        if self.diversity_alpha <= 0.0 or len(candidates) <= 1:
            best = candidates[0]
        else:
            best = max(candidates, key=lambda item: item[1] - self.diversity_alpha * use_count.get(key_fn(item[0]), 0))
        key = key_fn(best[0])
        use_count[key] = use_count.get(key, 0) + 1
        return best

    def run_hybrid_loop(self, start_image_id: Optional[str] = None, loops: int = 16) -> List[HybridStepRecord]:
        log: List[HybridStepRecord] = []
        all_image_ids = self.index.image_ids()
        if self.allowed_image_ids is not None:
            all_image_ids = [i for i in all_image_ids if i in self.allowed_image_ids]
        if not all_image_ids:
            return log

        if start_image_id is None or start_image_id not in all_image_ids:
            start_image_id = secrets.choice(all_image_ids)

        visited_image_ids: Set[str] = {start_image_id}
        visited_doc_ids:   Set[str] = set()
        visited_patch_ids: Set[str] = set()

        pool = self.index.image_patches_of(start_image_id)
        obj_pool = [p for p in pool if p.type == "object"]
        cur_patch = secrets.choice(obj_pool if obj_pool else pool)
        visited_patch_ids.add(cur_patch.patch_id)

        log.append(HybridStepRecord(step=0, action="start", modality="image",
                                     image_id=start_image_id, patch_id=cur_patch.patch_id,
                                     bbox=cur_patch.bbox, similarity_mode="semantic"))
        step_n = 1

        res = self.index.query_text(cur_patch.embedding, exclude_doc_ids=visited_doc_ids,
                                     min_win_size=self.min_win_size, threshold=self.threshold, top_k=self.top_k)
        if not res:
            res = self.index.query_text(cur_patch.embedding, exclude_doc_ids=visited_doc_ids,
                                         min_win_size=self.min_win_size, threshold=-1.0, top_k=self.top_k)
        if not res:
            return log

        tgt_txt, sim0 = self._pick_diverse(res, self.doc_use_count, lambda p: p.doc_id)
        visited_doc_ids.add(tgt_txt.doc_id)
        visited_patch_ids.add(tgt_txt.patch_id)
        hn_txt = self.index.find_hard_negative_text(tgt_txt, visited_doc_ids,
                                                      hardness_min=self.hardness_min,
                                                      hard_neg_sem_thresh=self.hard_neg_sem_thresh)
        log.append(HybridStepRecord(step=step_n, action="img_to_txt", modality="text",
                                     doc_id=tgt_txt.doc_id, patch_id=tgt_txt.patch_id,
                                     text_content=tgt_txt.text_content, similarity=sim0, similarity_mode="semantic",
                                     hard_negative_txt_patch_id=hn_txt[0] if hn_txt else None,
                                     hard_negative_txt_sim=hn_txt[1] if hn_txt else None,
                                     hard_negative_txt_mode=hn_txt[2] if hn_txt else None))
        step_n += 1

        for lp in range(1, loops + 1):
            txt_pool = [p for p in self.index.text_patches_of(tgt_txt.doc_id)
                        if p.patch_id not in visited_patch_ids and p.win_size >= self.min_win_size
                        and self.index._is_quality_text(p.text_content)]
            if not txt_pool:
                txt_pool = [p for p in self.index.text_patches_of(tgt_txt.doc_id) if p.patch_id not in visited_patch_ids]
            if txt_pool:
                tgt_txt = max(txt_pool, key=lambda p: self._cosine_sim(tgt_txt.embedding, p.embedding))
                visited_patch_ids.add(tgt_txt.patch_id)
                log.append(HybridStepRecord(step=step_n, action="txt_shift", modality="text",
                                             doc_id=tgt_txt.doc_id, patch_id=tgt_txt.patch_id,
                                             text_content=tgt_txt.text_content, similarity_mode="semantic"))
                step_n += 1

            # v5d: ziyaret edilenlere EK olarak, izin verilen havuzun DISINDAKI
            # tum gorselleri de haric tut -- boylece train sekansi test
            # gorseline (ya da tam tersi) SICRAYAMAZ.
            img_res = self.index.query_image(tgt_txt.embedding, exclude_image_ids=visited_image_ids | self._forbidden_image_ids, top_k=self.top_k)
            if not img_res:
                break
            best_img, best_score = self._pick_diverse(img_res, self.image_use_count, lambda p: p.image_id)
            visited_image_ids.add(best_img.image_id)
            visited_patch_ids.add(best_img.patch_id)
            log.append(HybridStepRecord(step=step_n, action="txt_to_img", modality="image",
                                         image_id=best_img.image_id, patch_id=best_img.patch_id,
                                         bbox=best_img.bbox, similarity=best_score, similarity_mode="semantic"))
            step_n += 1
            cur_patch = best_img

            MIN_OBJ = 0.4
            img_pool_raw = self.index.image_patches_of(cur_patch.image_id)
            valid_pool = [p for p in img_pool_raw if p.patch_id not in visited_patch_ids
                          and cur_patch.iou(p) <= self.max_iou_shift and p.objectness_score >= MIN_OBJ]
            if not valid_pool:
                valid_pool = sorted([p for p in img_pool_raw if p.patch_id not in visited_patch_ids],
                                     key=lambda x: x.objectness_score, reverse=True)[:5]

            if valid_pool:
                best_mode, best_patch = None, None
                best_surface_sim, best_sem_sim, best_hardness = 0.0, 0.0, -999.0
                for m in ("color", "shape", "texture"):
                    q_feat = self._get_feat(cur_patch, m)
                    if q_feat is None:
                        continue
                    candidate = max(valid_pool, key=lambda p: self._cosine_sim(q_feat, self._get_feat(p, m), fallback=-1.0))
                    surface_sim = self._cosine_sim(q_feat, self._get_feat(candidate, m), fallback=-1.0)
                    sem_sim = self._cosine_sim(cur_patch.embedding, candidate.embedding)
                    # v5: hardness karsilastirmasi z-normalize edilmis surface_sim ile yapiliyor
                    # (raporlanan best_surface_sim yine HAM deger olarak kaydediliyor).
                    surface_sim_z = self._calibrated(surface_sim, m)
                    hardness = surface_sim_z - sem_sim
                    if hardness > best_hardness:
                        best_mode, best_patch = m, candidate
                        best_surface_sim, best_sem_sim, best_hardness = surface_sim, sem_sim, hardness

                # NOT: best_hardness artik z-skor farkindan geldigi icin, hardness_min
                # esigi de v5'te yeniden kalibre edilmis olmali (asagidaki HUCRE E1'e bak).
                if best_patch is not None and best_hardness >= self.hardness_min:
                    mode = best_mode
                    tgt_img = best_patch
                    score = best_surface_sim
                    computed_label = "hard_negative" if best_sem_sim < self.hard_neg_sem_thresh else "positive"
                else:
                    mode = "semantic"
                    tgt_img = secrets.choice(valid_pool)
                    score = self._cosine_sim(cur_patch.embedding, tgt_img.embedding)
                    computed_label = "positive"
            else:
                mode = "semantic"
                score = 0.0
                computed_label = "positive"
                others = [p for p in img_pool_raw if p.patch_id != cur_patch.patch_id]
                tgt_img = min(others, key=lambda p: cur_patch.iou(p)) if others else cur_patch

            visited_patch_ids.add(tgt_img.patch_id)
            log.append(HybridStepRecord(step=step_n, action="img_shift", modality="image",
                                         image_id=tgt_img.image_id, patch_id=tgt_img.patch_id,
                                         bbox=tgt_img.bbox, similarity=score,
                                         similarity_mode=mode, verified_label=computed_label))
            step_n += 1
            cur_patch = tgt_img

            res = self.index.query_text(cur_patch.embedding, exclude_doc_ids=visited_doc_ids,
                                         min_win_size=self.min_win_size, threshold=self.threshold, top_k=self.top_k)
            if not res:
                res = self.index.query_text(cur_patch.embedding, exclude_doc_ids=visited_doc_ids,
                                             min_win_size=self.min_win_size, threshold=-1.0, top_k=self.top_k)
            if not res:
                break
            tgt_txt, sim = self._pick_diverse(res, self.doc_use_count, lambda p: p.doc_id)
            visited_doc_ids.add(tgt_txt.doc_id)
            visited_patch_ids.add(tgt_txt.patch_id)
            hn_txt = self.index.find_hard_negative_text(tgt_txt, visited_doc_ids,
                                                          hardness_min=self.hardness_min,
                                                          hard_neg_sem_thresh=self.hard_neg_sem_thresh)
            log.append(HybridStepRecord(step=step_n, action="img_to_txt", modality="text",
                                         doc_id=tgt_txt.doc_id, patch_id=tgt_txt.patch_id,
                                         text_content=tgt_txt.text_content, similarity=sim, similarity_mode="semantic",
                                         hard_negative_txt_patch_id=hn_txt[0] if hn_txt else None,
                                         hard_negative_txt_sim=hn_txt[1] if hn_txt else None,
                                         hard_negative_txt_mode=hn_txt[2] if hn_txt else None))
            step_n += 1

        return log


In [ ]:
# HÜCRE E2: esigi sabitle
# ONEMLI: 0.1 degeri 2K kosusundan DEVRALINMISTIR ve bu notebook'ta
# DEGISTIRILMEMELIDIR. 2K/10K karsilastirmasinin kontrollu kalmasi buna baglidir.
# E1 taramasi yalnizca teshis icindir; bandi tutturmak amac degildir.
HARDNESS_MIN = 0.1  # 2K kosusuyla AYNI -- degistirme
HARD_NEG_SEM_THRESH = 0.85  # v4'teki gibi kalsın, sem_sim ölçeği değişmedi
print(f"HARDNESS_MIN={HARDNESS_MIN}, HARD_NEG_SEM_THRESH={HARD_NEG_SEM_THRESH} olarak ayarlandı.")


## 3. COCO caption dosyası ve eşleme (önbellekten)

In [ ]:
# HÜCRE T2: COCO caption dosyasını arama
import os, json

candidate_paths = [
    f"{DRIVE_BASE}/captions_train2017.json",
    f"{DRIVE_BASE}/annotations/captions_train2017.json",
    f"{DRIVE_BASE}/coco/annotations/captions_train2017.json",
    f"{DRIVE_BASE}/ZoomInOut/captions_train2017.json",
    f"{DRIVE_BASE}/captions_val2017.json",
    f"{DRIVE_BASE}/annotations/captions_val2017.json",
    "/content/drive/MyDrive/captions_train2017.json",
    "/content/drive/MyDrive/annotations/captions_train2017.json",
]
COCO_CAPTIONS_PATH = None
for p in candidate_paths:
    if os.path.exists(p):
        COCO_CAPTIONS_PATH = p
        print(f"✅ BULUNDU: {p}")
        break

if COCO_CAPTIONS_PATH is None:
    print("❌ Standart yollarda bulunamadı.")
    print("Elinde bir COCO caption dosyası (captions_train2017.json /")
    print("captions_val2017.json, ya da Türkçe çevirisi -- ayni formatta")
    print("{\"annotations\": [{\"image_id\":.., \"caption\":..}, ...]} olmali)")
    print("varsa, aşağıya elle yolu yaz:")
    print('  COCO_CAPTIONS_PATH = "/content/drive/MyDrive/.../captions_train2017.json"')
    print("\nYoksa bu notebook otomatik olarak v4 davranışına (en-yakın-hikaye-")
    print("cümlesi) düşecek -- sadece Bölüm 2-4'teki (CV özellik zenginleştirme +")
    print("kalibrasyon) düzeltmeler uygulanacak.")


In [ ]:
# HÜCRE T3: Eşleştirme denemesi (COCO_CAPTIONS_PATH bulunduysa/ayarlandiysa)
image_id_to_captions = {}
COCO_MAPPING_READY = False
ID_FIELD = "image_id"  # <- doğrulandı: ds_sampled_image içindeki image_id, gerçek COCO image_id'siyle birebir aynı

if COCO_CAPTIONS_PATH and os.path.exists(COCO_CAPTIONS_PATH):
    with open(COCO_CAPTIONS_PATH) as f:
        coco_ann = json.load(f)
    for ann in coco_ann["annotations"]:
        image_id_to_captions.setdefault(ann["image_id"], []).append(ann["caption"])
    print(f"COCO caption dosyası yüklendi: {len(image_id_to_captions)} benzersiz görüntü, "
          f"{len(coco_ann['annotations'])} toplam caption.")

    def normalize_coco_id(raw):
        # COCO id'leri hem int hem "000000123456.jpg" formatinda olabilir
        if isinstance(raw, int):
            return raw
        s = str(raw)
        s = s.split(".")[0].lstrip("0") or "0"
        try:
            return int(s)
        except ValueError:
            return None

    if ID_FIELD is None:
        print("\n⚠️  ID_FIELD ayarlanmadı. Hücre T1'in çıktısına bakıp bu hücrenin")
        print("    başındaki ID_FIELD değişkenini ayarla, sonra bu hücreyi tekrar çalıştır.")
    else:
        n_total = len(ds_sampled_image)
        n_matched = 0
        sample_matches = []
        for i in range(n_total):
            raw_id = ds_sampled_image[i][ID_FIELD]
            norm_id = normalize_coco_id(raw_id)
            if norm_id in image_id_to_captions:
                n_matched += 1
                if len(sample_matches) < 5:
                    sample_matches.append((i, norm_id, image_id_to_captions[norm_id][0]))

        print(f"\nEşleşme oranı: {n_matched}/{n_total} (%{n_matched/n_total*100:.1f})")
        print("\nÖrnek eşleşmeler:")
        for i, cid, cap in sample_matches:
            print(f"  ds_sampled_image[{i}] (coco_id={cid}) -> {cap!r}")

        if n_matched / n_total > 0.5:
            COCO_MAPPING_READY = True
            print("\n✅ Eşleşme oranı yeterli (%50+) -- COCO caption'ları kullanılacak.")
        else:
            print("\n⚠️  Eşleşme oranı düşük -- ID_FIELD yanlış olabilir, kontrol et.")
            print("    COCO_MAPPING_READY=False kalacak, v4 davranışına düşülecek.")
else:
    print("COCO_CAPTIONS_PATH yok -- v4 davranışına (en-yakın-hikaye-cümlesi) düşülüyor.")

print(f"\nCOCO_MAPPING_READY = {COCO_MAPPING_READY}")


In [ ]:
# HÜCRE C0: coco_image_N -> orijinal COCO id eşleme tablosu
patch_image_id_to_coco_id = {}
if COCO_MAPPING_READY:
    for i in range(len(ds_sampled_image)):
        raw_id = ds_sampled_image[i][ID_FIELD]
        norm_id = normalize_coco_id(raw_id)
        patch_image_id_to_coco_id[f"coco_image_{i}"] = norm_id
    print(f"✅ {len(patch_image_id_to_coco_id)} görüntü için coco_id eşlemesi hazır.")
else:
    print("COCO_MAPPING_READY=False -- eşleme tablosu boş bırakıldı.")


In [ ]:
# HÜCRE C1: COCO caption\'larını Qwen3-VL-Embedding-2B ile embed et
coco_caption_embeddings = {}  # {coco_id: [(caption_text, embedding), ...]}

if COCO_MAPPING_READY:
    if os.path.exists(COCO_CAPTIONS_EMB_PATH):
        print(f"⏳ Önceden embed edilmiş COCO caption\'ları bulundu, yükleniyor: {COCO_CAPTIONS_EMB_PATH}")
        with open(COCO_CAPTIONS_EMB_PATH, "rb") as f:
            coco_caption_embeddings = pickle.load(f)
    else:
        print("⏳ Qwen3-VL-Embedding-2B yükleniyor (GPU gerektirir, biraz sürer)...")
        from transformers import AutoModel, AutoProcessor

        _qwen_model_id = "Qwen/Qwen3-VL-Embedding-2B"
        _qwen_model = AutoModel.from_pretrained(_qwen_model_id, trust_remote_code=True,
                                                 torch_dtype=torch.float16).to(device)
        _qwen_processor = AutoProcessor.from_pretrained(_qwen_model_id, trust_remote_code=True)
        print("✅ Model yüklendi.")

        def _last_token_pool(last_hidden_states, attention_mask):
            left_padding = (attention_mask[:, -1].sum() == attention_mask.shape[0])
            if left_padding:
                return last_hidden_states[:, -1]
            seq_lens = attention_mask.sum(dim=1) - 1
            batch_size = last_hidden_states.shape[0]
            return last_hidden_states[torch.arange(batch_size, device=last_hidden_states.device), seq_lens]

        def _embed_texts(texts, batch_size=32):
            all_embs = []
            for start in range(0, len(texts), batch_size):
                batch = texts[start:start + batch_size]
                inputs = _qwen_processor(text=batch, padding=True, truncation=True,
                                          max_length=64, return_tensors="pt").to(device)
                with torch.no_grad(), torch.autocast(device_type="cuda", dtype=torch.bfloat16):
                    out = _qwen_model(input_ids=inputs.input_ids, attention_mask=inputs.attention_mask,
                                       output_hidden_states=True, return_dict=True)
                    last_hidden = out.hidden_states[-1] if hasattr(out, "hidden_states") else out[0]
                    emb = _last_token_pool(last_hidden.float(), inputs.attention_mask)
                    emb = F.normalize(emb, p=2, dim=-1)
                all_embs.append(emb.cpu().numpy())
            return np.concatenate(all_embs, axis=0)

        # SADECE bizim 2000 örneklem görselimizin caption'larını embed et
        # (image_id_to_captions TÜM COCO'yu -- 118K görsel -- içeriyor, gereksiz
        # hesaplamadan kaçınmak için patch_image_id_to_coco_id ile filtrele).
        _needed_coco_ids = set(patch_image_id_to_coco_id.values())
        all_pairs_flat = [(coco_id, cap) for coco_id, caps in image_id_to_captions.items()
                           if coco_id in _needed_coco_ids for cap in caps]
        all_texts_flat = [p[1] for p in all_pairs_flat]
        print(f"({len(image_id_to_captions)} COCO görselinden sadece {len(_needed_coco_ids)} tanesi "
              f"bizim örneklemimizde -- {len(all_texts_flat)} caption embed edilecek, "
              f"591753 değil.)")
        print(f"⏳ {len(all_texts_flat)} caption embed ediliyor...")
        all_embs_flat = _embed_texts(all_texts_flat, batch_size=64)

        for (coco_id, cap), emb in zip(all_pairs_flat, all_embs_flat):
            coco_caption_embeddings.setdefault(coco_id, []).append((cap, emb))

        with open(COCO_CAPTIONS_EMB_PATH, "wb") as f:
            pickle.dump(coco_caption_embeddings, f)
        print(f"✅ Kaydedildi -> {COCO_CAPTIONS_EMB_PATH}")

        del _qwen_model, _qwen_processor
        torch.cuda.empty_cache()

    print(f"\n{len(coco_caption_embeddings)} görüntü için caption embedding\'i hazır.")
else:
    print("COCO_MAPPING_READY=False -- bu hücre atlanıyor, v4 davranışına (en-yakın-hikaye-cümlesi) devam edilecek.")


## 4. Genişletilmiş havuz (ext görseller, önbellekten)

In [ ]:
# HÜCRE EXT1: yeni ham veri kumesi + embedding pkl'ini yukle, GERCEK COCO ID uzerinden cakisma kontrolu
NEW_IMG_EMB_PKL = f"{DRIVE_BASE}/ZoomInOut/coco_image_dynamic_embeddings_new_WH_greater_than_224_sample_size_10000_2048.pkl"
NEW_RAW_IMAGE_SOURCE = f"{DRIVE_BASE}/ZoomInOut/ds_sampled_image_datas_WH_greater_than_224_sample_size_10000"

with open(NEW_IMG_EMB_PKL, "rb") as f:
    new_image_metadata = pickle.load(f)
print(f"Yeni embedding dosyasinda toplam patch sayisi: {len(new_image_metadata)}")

# KRITIK: arkadasin image_id/patch_id degerleri BIZIM KENDI "coco_image_N" isim
# uzayimizla CAKISIYOR OLABILIR (o da ayni formati kullaniyor, N=0..9999 -- ama
# N=0..1999 araliginda bizim 47. gorselimizle onun 47. gorseli TAMAMEN FARKLI
# fotograflar olabilir, sadece string ayni). Her sey baslamadan HEMEN "ext_"
# on-ekiyle ayri bir isim uzayina tasiyoruz -- boylece hybrid_index'te ASLA
# cakismaz.
for m in new_image_metadata:
    m["image_id"] = "ext_" + m["image_id"]
    m["patch_id"] = "ext_" + m["patch_id"]
print(f"image_id/patch_id 'ext_' on-ekiyle ayri isim uzayina tasindi (cakisma onlendi).")

new_raw_dataset = load_from_disk(NEW_RAW_IMAGE_SOURCE)
print(f"Yeni ham veri kumesi yuklendi: {len(new_raw_dataset)} kayit, sutunlar: {new_raw_dataset.column_names}")
assert ID_FIELD in new_raw_dataset.column_names, f"BEKLENMEYEN SEMA: '{ID_FIELD}' sutunu yok, arkadasa sor."

# pozisyonel indeks -> GERCEK COCO id eslemesi (bizim patch_image_id_to_coco_id ile AYNI mantik,
# sadece yeni ham veri kumesi icin -- HUCRE C0'a bak)
new_idx_to_coco_id = {}
for i in range(len(new_raw_dataset)):
    raw_id = new_raw_dataset[i][ID_FIELD]
    new_idx_to_coco_id[i] = normalize_coco_id(raw_id)

print(f"\nOrnek eslemeler (pozisyon -> gercek COCO id):")
for i in [0, 1, 10, 100, 1000]:
    if i < len(new_raw_dataset):
        print(f"  index {i} -> coco_id {new_idx_to_coco_id.get(i)}")

def _extract_pos_idx(image_id):
    return int(str(image_id).split("_")[-1])

new_image_ids = sorted(set(m["image_id"] for m in new_image_metadata))
print(f"\nBenzersiz yeni gorsel (patch kaynagi) sayisi: {len(new_image_ids)}")
print(f"Gorsel basina ort. patch: {len(new_image_metadata)/len(new_image_ids):.1f}")

new_coco_id_map = {}  # coco_numeric_id -> arkadasin kendi image_id'si (coco_image_N)
for iid in new_image_ids:
    pos_idx = _extract_pos_idx(iid)
    coco_id = new_idx_to_coco_id.get(pos_idx)
    if coco_id is not None:
        new_coco_id_map[coco_id] = iid

print(f"{len(new_coco_id_map)}/{len(new_image_ids)} gorsel gercek COCO id'ye basariyla cevrildi.")

existing_coco_ids = set(patch_image_id_to_coco_id.values()) if COCO_MAPPING_READY else set()
print(f"\nBizim mevcut havuzumuzun gercek COCO numeric ID sayisi: {len(existing_coco_ids)}")

overlap_coco_ids = existing_coco_ids & set(new_coco_id_map.keys())
print(f"GERCEK ICERIK cakismasi (ayni fotograf): {len(overlap_coco_ids)}/{len(existing_coco_ids)}")
if overlap_coco_ids:
    print("-> Beklenen durum (arkadasin 10K'si eski 2000'i de iceriyor). Sorun DEGIL --")
    print("   asagida OTOMATIK filtrelenip sadece GERCEKTEN YENI olanlar kalacak.")

truly_new_coco_ids = set(new_coco_id_map.keys()) - existing_coco_ids
truly_new_ids = set(new_coco_id_map[cid] for cid in truly_new_coco_ids)
print(f"\nGERCEKTEN YENI (bizde olmayan) gorsel sayisi: {len(truly_new_ids)}")

new_image_ids = sorted(truly_new_ids)
new_image_metadata = [m for m in new_image_metadata if m["image_id"] in truly_new_ids]
print(f"new_image_metadata GUNCELLENDI: artik {len(new_image_metadata)} patch, "
      f"{len(new_image_ids)} gercekten-yeni gorsele ait (EXT2/EXT3 bunu kullanacak).")

# EXT4'te kullanilacak: yeni gorsel -> gercek coco_id eslemesi
new_patch_image_id_to_coco_id = {
    orig_iid: cid for cid, orig_iid in new_coco_id_map.items() if cid in truly_new_coco_ids
}


In [ ]:
# HÜCRE EXT2: ham piksellere erisim -- DOGRUDAN Drive'daki yeni veri kumesinden
# (arkadasin dosyasi ham piksel de icerdigi icin COCO'dan AYRICA INDIRMEYE GEREK YOK)
new_pos_idx_lookup = {iid: _extract_pos_idx(iid) for iid in new_image_ids}

def ext_visual_loader(image_id):
    idx = new_pos_idx_lookup[image_id]
    return new_raw_dataset[idx]["image"].convert("RGB")

print(f"Ham piksel erisimi hazir -- {len(new_image_ids)} gorsel icin dogrudan "
      f"'{NEW_RAW_IMAGE_SOURCE}' kullanilacak (indirme yok, hepsi zaten Drive'da).")


In [ ]:
# HÜCRE EXT3: CV ozellik cikarimi (onbellekli; spot-check ana notebookta yapildi) -- tam CV ozellik cikarimi
EXT_CV_CACHE_PATH = f"{DRIVE_BASE}/ZoomInOut/v5i_02_cv_feature_cache.pkl"  # YENI isim, eskiyle cakismaz

from skimage.feature import local_binary_pattern

if os.path.exists(EXT_CV_CACHE_PATH):
    print(f"Onbellek zaten var, yukleniyor: {EXT_CV_CACHE_PATH}")
    with open(EXT_CV_CACHE_PATH, "rb") as f:
        ext_cv_cache = pickle.load(f)
else:
    print("Onbellek yok -- NATIVE cozunurlukte (olceklemesiz) CV ozellikleri hesaplaniyor...")
    ext_cv_cache = {}
    for m in tqdm(new_image_metadata, desc="Yeni gorseller icin CV ozellikleri"):
        try:
            pil_img = ext_visual_loader(m["image_id"])
            img_bgr = cv2.cvtColor(np.array(pil_img.convert("RGB")), cv2.COLOR_RGB2BGR)
            H_img, W_img = img_bgr.shape[:2]
            bbox = m["bbox"]
            # NATIVE bbox -- 224 OLCEKLEMESI YOK, dogrudan kullan
            x1 = int(np.clip(bbox[0], 0, W_img - 1))
            y1 = int(np.clip(bbox[1], 0, H_img - 1))
            x2 = int(np.clip(bbox[2], 1, W_img))
            y2 = int(np.clip(bbox[3], 1, H_img))
            if x2 <= x1: x2 = x1 + 1
            if y2 <= y1: y2 = y1 + 1
            crop = img_bgr[y1:y2, x1:x2]

            hsv = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)
            hist_h = cv2.calcHist([hsv], [0], None, [16], [0, 180])
            hist_s = cv2.calcHist([hsv], [1], None, [8], [0, 256])
            hist_v = cv2.calcHist([hsv], [2], None, [8], [0, 256])
            color_f = np.concatenate([hist_h, hist_s, hist_v]).flatten()
            color_f /= (np.linalg.norm(color_f) + 1e-10)

            gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
            sobelx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
            sobely = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
            mag = np.sqrt(sobelx**2 + sobely**2)
            hist_txt, _ = np.histogram(mag, bins=32, range=(0, 255))
            hist_txt = hist_txt.astype(np.float32)
            lbp = local_binary_pattern(gray, P=8, R=1, method="uniform")
            hist_lbp, _ = np.histogram(lbp, bins=10, range=(0, 10))
            hist_lbp = hist_lbp.astype(np.float32)
            texture_f = np.concatenate([hist_txt, hist_lbp])
            texture_f /= (np.linalg.norm(texture_f) + 1e-10)

            edges = cv2.Canny(gray, 50, 150)
            hist_edge, _ = np.histogram(edges, bins=32, range=(0, 255))
            moments = cv2.moments(edges)
            hu = cv2.HuMoments(moments).flatten()
            hu_log = -np.sign(hu) * np.log10(np.abs(hu) + 1e-10)
            hu_log = np.nan_to_num(hu_log, nan=0.0, posinf=0.0, neginf=0.0)
            aspect_ratio = float(x2 - x1) / float(y2 - y1)
            shape_f = np.concatenate([hist_edge.astype(np.float32), hu_log.astype(np.float32), [aspect_ratio]])
            shape_f /= (np.linalg.norm(shape_f) + 1e-10)

            ext_cv_cache[m["patch_id"]] = (color_f, shape_f, texture_f)
        except Exception as e:
            continue

    with open(EXT_CV_CACHE_PATH, "wb") as f:
        pickle.dump(ext_cv_cache, f)
    print(f"Kaydedildi -> {EXT_CV_CACHE_PATH} ({len(ext_cv_cache)}/{len(new_image_metadata)} basarili)")


In [ ]:
# HÜCRE EXT4: yeni gorseller icin GERCEK COCO caption'larini embed et
print(f"{len(new_patch_image_id_to_coco_id)} yeni gorsel icin coco_id eslemesi hazir (EXT1'de uretildi).")

missing_caption_coco_ids = [cid for cid in new_patch_image_id_to_coco_id.values()
                             if cid not in coco_caption_embeddings and cid in image_id_to_captions]
no_caption_at_all = [cid for cid in new_patch_image_id_to_coco_id.values() if cid not in image_id_to_captions]
print(f"Caption'i henuz embed edilmemis {len(missing_caption_coco_ids)} gorsel var, embed edilecek.")
if no_caption_at_all:
    print(f"UYARI: {len(no_caption_at_all)} yeni gorsel icin COCO caption dosyasinda HIC caption yok "
          f"(bu gorseller COCO_MAPPING_READY kapsaminda degil olabilir) -- bunlar atlanacak.")

if missing_caption_coco_ids:
    if "_embed_texts" not in globals():
        print("Qwen modeli bu oturumda hic yuklenmemis (onbellekten yuklendigi icin), simdi yukleniyor...")
        import subprocess, sys
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "accelerate"], check=False)
        import gc
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        from transformers import AutoModel, AutoProcessor
        _qwen_model_id = "Qwen/Qwen3-VL-Embedding-2B"
        # ONEMLI: device_map="auto" + low_cpu_mem_usage=True -- model agirliklari
        # CPU RAM'de tam materyalize edilmeden DOGRUDAN GPU'ya yukleniyor. Bu
        # olmadan (.to(device) ile) Colab'in standart RAM'i (~12GB) tukeniyor ve
        # oturum coküyor -- tam bu hatayi yasadik.
        _qwen_model = AutoModel.from_pretrained(_qwen_model_id, trust_remote_code=True,
                                                 torch_dtype=torch.float16,
                                                 device_map="auto", low_cpu_mem_usage=True)
        _qwen_processor = AutoProcessor.from_pretrained(_qwen_model_id, trust_remote_code=True)

        def _last_token_pool(last_hidden_states, attention_mask):
            left_padding = (attention_mask[:, -1].sum() == attention_mask.shape[0])
            if left_padding:
                return last_hidden_states[:, -1]
            seq_lens = attention_mask.sum(dim=1) - 1
            batch_size = last_hidden_states.shape[0]
            return last_hidden_states[torch.arange(batch_size, device=last_hidden_states.device), seq_lens]

        def _embed_texts(texts, batch_size=32):
            all_embs = []
            for start in range(0, len(texts), batch_size):
                batch = texts[start:start + batch_size]
                inputs = _qwen_processor(text=batch, padding=True, truncation=True,
                                          max_length=64, return_tensors="pt").to(device)
                with torch.no_grad(), torch.autocast(device_type="cuda", dtype=torch.bfloat16):
                    out = _qwen_model(input_ids=inputs.input_ids, attention_mask=inputs.attention_mask,
                                       output_hidden_states=True)
                    pooled = _last_token_pool(out.hidden_states[-1], inputs.attention_mask)
                all_embs.append(pooled.float().cpu().numpy())
            return np.concatenate(all_embs, axis=0)

    for cid in tqdm(missing_caption_coco_ids, desc="Yeni caption embedding"):
        caps = image_id_to_captions[cid]
        embs = _embed_texts(caps)
        coco_caption_embeddings[cid] = list(zip(caps, embs))

    # ONEMLI: ESKI v5_coco_captions_embedded.pkl EZILMIYOR -- YENI isimle kaydediliyor.
    MERGED_COCO_CAPTIONS_PATH = f"{DRIVE_BASE}/ZoomInOut/v5i_02_coco_captions_embedded_merged.pkl"
    with open(MERGED_COCO_CAPTIONS_PATH, "wb") as f:
        pickle.dump(coco_caption_embeddings, f)
    print(f"Guncellendi ve YENI isimle kaydedildi -> {MERGED_COCO_CAPTIONS_PATH} "
          f"(artik toplam {len(coco_caption_embeddings)} gorselin caption'i var)")
else:
    print("Tum yeni gorsellerin caption'lari zaten hazir.")


In [ ]:
# HÜCRE EXT5: yeni patch'leri (CV ozellikleriyle) ana hybrid_index'e ekle
n_added = 0
n_no_cv = 0
for m in new_image_metadata:
    feats = ext_cv_cache.get(m["patch_id"])
    if feats is not None:
        color_f, shape_f, texture_f = feats
    else:
        color_f = shape_f = texture_f = None
        n_no_cv += 1
    hybrid_index.add_image_patch(ImagePatchMeta(
        image_id=m["image_id"], patch_id=m["patch_id"], bbox=m["bbox"],
        embedding=m["embedding"], objectness_score=m.get("objectness_score", 1.0),
        type=m.get("type", "object"),
        color_feat=color_f, shape_feat=shape_f, texture_feat=texture_f,
    ))
    patch_image_id_to_coco_id[m["image_id"]] = new_patch_image_id_to_coco_id.get(m["image_id"])
    n_added += 1

print(f"{n_added} yeni patch hybrid_index'e eklendi ({n_no_cv} tanesinin CV ozelligi yoktu, embedding ile dolduruldu).")
print(f"YENI toplam benzersiz gorsel sayisi: {len(hybrid_index.image_ids())} "
      f"(onceki: 2000 + gercekten-yeni: {len(new_image_ids)})")
print(f"YENI toplam patch sayisi: {len(hybrid_index.image_patches)}")

print("\nSIRADAKI ADIM: yukaridaki '## 8. Ayarlar + train/test split ...' (HUCRE 6) ve")
print("'## 9. Train + val + test uretimi' (HUCRE 7) hucrelerini SIRAYLA tekrar calistir --")
print("artik v5i_02_* path'lerine yazacaklar ve bu GENISLETILMIS havuzdan sifirdan uretecekler.")
print("Ardindan Bolum 24'teki (D_SACL_FOCUS1-3) hucreleri tekrar calistirarak hocanin")
print("istedigi 5-seed/epoch/sure deneyini bu genisletilmis veriyle tekrarla.")


## 5. Ayarlar, split-koruyan bölme ve indeks hızlandırma

In [ ]:
# HÜCRE 6: Ayarlar + train/test split + tüm üretim fonksiyonları

NUM_SEQUENCES        = 600  # v5e: 300->600, cesitlilik + hacim icin
LOOPS_PER_SEQUENCE   = 16
MIN_CHAIN_LENGTH     = 16
SIM_THRESHOLD        = 0.55
TRAIN_RATIO          = 0.75  # v5e: test havuzunu 400->500 gorsele cikardik (cesitlilik icin)

TEST_NUM_SEQUENCES   = 600   # v5e: 200->600, daha fazla benzersiz gorsele ulasma sansi icin
TEST_TARGET_SAMPLES  = 400   # v5e: 500->400, 500 gorselik havuzda gercekci/ulasilabilir hedef

VAL_RATIO            = 0.15  # v5h: train havuzunun %15'i erken-durdurma/model-secimi icin ayrilir
VAL_NUM_SEQUENCES    = 300
VAL_TARGET_SAMPLES   = 150

TOP_K              = 8     # GERI DONULDU: v5i degeri (v5j denemesi 14 idi)
DIVERSITY_ALPHA    = 0.03  # GERI DONULDU: v5i degeri (v5j denemesi 0.06 idi)


def make_image_split(hybrid_index, train_ratio=0.8, seed=42):
    rng = random.Random(seed)
    all_ids = hybrid_index.image_ids()
    rng.shuffle(all_ids)
    split_point = int(len(all_ids) * train_ratio)
    return set(all_ids[:split_point]), set(all_ids[split_point:])


def carve_val_split(train_image_ids, val_ratio=0.15, seed=43):
    """v5h: train havuzunun bir kismini erken-durdurma / model-secimi icin
    ayirir. Test'e HIC dokunmaz -- sadece train'i (final_train, val) olarak
    ikiye boler. sorted() ile deterministik (set sirasina guvenmiyoruz)."""
    rng = random.Random(seed)
    all_ids = sorted(train_image_ids)
    rng.shuffle(all_ids)
    split_point = int(len(all_ids) * (1 - val_ratio))
    return set(all_ids[:split_point]), set(all_ids[split_point:])


def generate_sequences(hybrid_index, engine, image_ids, num_sequences, loops, min_chain_length):
    """v5j: GARANTI-KAPSAMLI baslangic secimi -- eskiden her cekilis
    BAGIMSIZ rastgeleydi (random.choice), yani 600 cekiliste havuzun
    (1275) sadece ~%38'i (kupon toplama beklentisi) baslangic olarak
    kullanilirdi. Simdi havuz once KARISTIRILIP SIRAYLA gezilir -- ilk
    min(num_sequences, len(image_ids)) cekiliste HER goruntu TAM 1 KEZ
    baslangic olur (garantili), fazlasi icin bastan tekrar baslanir."""
    image_ids = list(image_ids)
    _rng_starts = random.Random(1234)
    shuffled = image_ids[:]
    _rng_starts.shuffle(shuffled)
    sequences = []
    for i in tqdm(range(num_sequences), desc="Sekans üretiliyor"):
        start_img = shuffled[i % len(shuffled)]
        chain = engine.run_hybrid_loop(start_image_id=start_img, loops=loops)
        if len(chain) >= min_chain_length:
            sequences.append(chain)
    print(f"✅ {len(sequences)}/{num_sequences} sekans üretildi (min_chain_length={min_chain_length} filtresinden geçti)")
    return sequences


def build_pairs_from_sequences(sequences, sim_threshold):
    """cross-modal pozitif çiftler + SADECE verified_label=='hard_negative' olan
    img_shift'lerden alınan hard negative'ler."""
    records = []
    n_with_hn = 0
    for chain in sequences:
        for i, step in enumerate(chain):
            if step.action not in ("img_to_txt", "txt_to_img") or i == 0:
                continue
            prev = chain[i - 1]
            img_step, txt_step = (prev, step) if step.action == "img_to_txt" else (step, prev)
            if img_step.modality != "image" or txt_step.modality != "text":
                continue
            cross_sim = step.similarity if step.similarity else 0.0
            if cross_sim < sim_threshold:
                continue

            record = {
                "txt_patch_id": txt_step.patch_id, "img_patch_id": img_step.patch_id,
                "cross_modal_sim": cross_sim,
                "hard_negative_patch_id": None, "hard_negative_sim": None, "hard_negative_mode": None,
                # Fikir 2: metin-tarafı hard-negative (sadece txt_step bir img_to_txt
                # adımıysa dolu olur -- txt_to_img çiftlerinde txt_step bir txt_shift
                # olduğu için bu alanlar None kalır, kapsam ~%50 civarında olur).
                "hard_negative_txt_patch_id": getattr(txt_step, "hard_negative_txt_patch_id", None),
                "hard_negative_txt_sim": getattr(txt_step, "hard_negative_txt_sim", None),
                "hard_negative_txt_mode": getattr(txt_step, "hard_negative_txt_mode", None),
            }
            if i + 1 < len(chain) and chain[i + 1].action == "img_shift":
                shift_step = chain[i + 1]
                if (shift_step.image_id == img_step.image_id
                        and getattr(shift_step, "verified_label", "positive") == "hard_negative"):
                    record["hard_negative_patch_id"] = shift_step.patch_id
                    record["hard_negative_sim"] = shift_step.similarity
                    record["hard_negative_mode"] = shift_step.similarity_mode
                    n_with_hn += 1
            records.append(record)

    print(f"Toplam cross-modal çift: {len(records)} | Doğrulanmış hard-negative içeren: {n_with_hn} (%{n_with_hn/max(len(records),1)*100:.1f})")
    return records


def attach_embeddings(records, hybrid_index):
    txt_lookup = {p.patch_id: p.embedding for p in hybrid_index.text_patches}
    img_lookup = {p.patch_id: p.embedding for p in hybrid_index.image_patches}
    img_id_lookup = {p.patch_id: p.image_id for p in hybrid_index.image_patches}  # v5: coco caption eşlemesi için
    final = []
    n_with_hn_txt = 0
    n_coco_caption_used = 0
    for r in records:
        txt_emb = txt_lookup.get(r["txt_patch_id"])
        img_emb = img_lookup.get(r["img_patch_id"])
        if txt_emb is None or img_emb is None:
            continue

        # v5: COCO caption bulunabiliyorsa, pozitif metni "en yakın hikaye
        # cümlesi" yerine görselin GERÇEK caption'ıyla değiştir.
        if COCO_MAPPING_READY:
            img_image_id = img_id_lookup.get(r["img_patch_id"])
            coco_id = patch_image_id_to_coco_id.get(img_image_id)
            if coco_id is not None and coco_id in coco_caption_embeddings:
                _cap_text, _cap_emb = random.choice(coco_caption_embeddings[coco_id])
                txt_emb = _cap_emb
                n_coco_caption_used += 1

        hn_emb = img_lookup.get(r["hard_negative_patch_id"]) if r["hard_negative_patch_id"] is not None else None
        hn_txt_emb = txt_lookup.get(r["hard_negative_txt_patch_id"]) if r.get("hard_negative_txt_patch_id") is not None else None
        if hn_txt_emb is not None:
            n_with_hn_txt += 1
        final.append({
            "txt_embedding": np.asarray(txt_emb, dtype=np.float32),
            "img_embedding": np.asarray(img_emb, dtype=np.float32),
            "cross_modal_sim": r["cross_modal_sim"],
            "hard_negative_embedding": np.asarray(hn_emb, dtype=np.float32) if hn_emb is not None else None,
            "hard_negative_sim": r["hard_negative_sim"],
            "hard_negative_mode": r["hard_negative_mode"],
            "hard_negative_txt_embedding": np.asarray(hn_txt_emb, dtype=np.float32) if hn_txt_emb is not None else None,
            "hard_negative_txt_sim": r.get("hard_negative_txt_sim"),
        })
    print(f"Embedding'i tam bulunan final kayıt: {len(final)} (metin-tarafı hard-negative içeren: {n_with_hn_txt}, %{n_with_hn_txt/max(len(final),1)*100:.1f})")
    if COCO_MAPPING_READY:
        print(f"COCO caption ile değiştirilen pozitif metin sayısı: {n_coco_caption_used} (%{n_coco_caption_used/max(len(final),1)*100:.1f})")
    return final


def generate_test_pairs(hybrid_index, engine, test_image_ids, num_sequences, loops,
                         min_chain_length, sim_threshold, target_samples):
    """v5f: TEST HAVUZU CESITLILIGI + DAGILIM TUTARLILIGI DUZELTMESI.
    Eskiden iki sorun vardi: (1) ayni gorsel test kumesinde defalarca
    tekrarlanabiliyordu (R@1'i yapay tavanliyordu -- v5e'de duzeltildi,
    her gorselden EN FAZLA 1 ornek); (2) SADECE dogrulanmis hard-negative
    iceren kayitlar tutuluyordu -- bu, test kumesini train'den (hard-negative
    orani ~%17) FARKLI bir dagilima (test'te %100) zorluyordu VE havuzu
    (17896 -> 2928 -> sadece 117 benzersiz gorsel) cok daralttiginin asil
    nedeniydi. Simdi test, train ile AYNI (filtresiz) dagilimdan orneklendigi
    icin havuz cok daha genis, benzersiz gorsele ulasmak cok daha kolay."""
    test_sequences = []
    test_image_ids = list(test_image_ids)
    _rng_starts = random.Random(1234)
    shuffled = test_image_ids[:]
    _rng_starts.shuffle(shuffled)
    for i in tqdm(range(num_sequences), desc="Test sekansları"):
        start_img = shuffled[i % len(shuffled)]
        chain = engine.run_hybrid_loop(start_image_id=start_img, loops=loops)
        if len(chain) >= min_chain_length:
            test_sequences.append(chain)

    raw_records = build_pairs_from_sequences(test_sequences, sim_threshold)
    random.shuffle(raw_records)

    img_id_lookup = {p.patch_id: p.image_id for p in hybrid_index.image_patches}
    seen_images = set()
    deduped_records = []
    for r in raw_records:
        src_img_id = img_id_lookup.get(r["img_patch_id"])
        if src_img_id in seen_images:
            continue
        seen_images.add(src_img_id)
        deduped_records.append(r)
        if len(deduped_records) >= target_samples:
            break

    if len(deduped_records) < target_samples:
        print(f"⚠️  UYARI: hedef {target_samples} idi ama sadece {len(deduped_records)} BENZERSİZ "
              f"görselden geçerli hard-negative çift bulunabildi. Test havuzunu (test_image_ids "
              f"oranı / num_sequences) büyütmen gerekebilir.")
    print(f"Benzersiz kaynak görsel sayısı (tekrarsız): {len(deduped_records)} / hedef {target_samples}")

    final_pairs = attach_embeddings(deduped_records, hybrid_index)
    mode_counts = {}
    for p in final_pairs:
        m = p["hard_negative_mode"]
        mode_counts[m] = mode_counts.get(m, 0) + 1
    print("Mod dağılımı:", mode_counts)
    return final_pairs


train_image_ids, test_image_ids = make_image_split(hybrid_index, TRAIN_RATIO)
train_image_ids, val_image_ids = carve_val_split(train_image_ids, VAL_RATIO, seed=43)
print(f"Train görsel sayısı: {len(train_image_ids)} | Val görsel sayısı: {len(val_image_ids)} | Test görsel sayısı: {len(test_image_ids)}")


In [ ]:
# HÜCRE PS1: 2K atamasını birebir koruyan bölme + AYRI cikti yollari
# ---------------------------------------------------------------------------
# NEDEN: v5i_02'de make_image_split, birlesik 10.000'lik indeksin TAMAMINI
# seed=42 ile karistiriyordu. Bu, orijinal 2000 gorseli de yeniden dagitiyor
# ve 2K/10K karsilastirmasini "ayni bolme + daha cok veri" olmaktan cikariyor.
# COZUM: 2K kosusundaki atama, YALNIZCA orijinal 2000 id'nin listesinin seed=42
# ile karistirilmasiyla olusmustu. Ayni listeyi ayni seed'le karistirmak o
# atamayi BIREBIR yeniden uretir. Yeni gorseller AYNI oranlarla ustune dagitilir.
# ---------------------------------------------------------------------------
_all_ids  = hybrid_index.image_ids()
_base_ids = [i for i in _all_ids if not str(i).startswith("ext_")]   # orijinal 2000
_new_ids  = [i for i in _all_ids if str(i).startswith("ext_")]       # eklenen 8000
print(f"Havuz: {len(_all_ids)} gorsel  =  {len(_base_ids)} orijinal + {len(_new_ids)} yeni")
assert len(_base_ids) > 0 and len(_new_ids) > 0, "ext_ on-ekli/on-eksiz gorseller ayirt edilemedi"

# --- (a) 2K atamasini BIREBIR yeniden uret (make_image_split + carve_val_split ile ayni mantik) ---
_rng = random.Random(42)
_shuf = _base_ids[:]                      # 2K kosusundaki ile AYNI ekleme sirasi
_rng.shuffle(_shuf)
_sp = int(len(_shuf) * TRAIN_RATIO)
_base_train_full, base_test = set(_shuf[:_sp]), set(_shuf[_sp:])

_rng2 = random.Random(43)
_bt = sorted(_base_train_full)            # carve_val_split sorted() kullaniyor
_rng2.shuffle(_bt)
_sp2 = int(len(_bt) * (1 - VAL_RATIO))
base_train, base_val = set(_bt[:_sp2]), set(_bt[_sp2:])
print(f"2K atamasi yeniden uretildi -> train={len(base_train)}  val={len(base_val)}  test={len(base_test)}")

# --- (b) yeni 8000 gorseli AYNI oranlarla ustune dagit ---
_rng3 = random.Random(4242)
_nw = _new_ids[:]
_rng3.shuffle(_nw)
_n_test = int(len(_nw) * (1 - TRAIN_RATIO))
new_test = set(_nw[:_n_test])
_rest = _nw[_n_test:]
_n_val = int(len(_rest) * VAL_RATIO)
new_val   = set(_rest[:_n_val])
new_train = set(_rest[_n_val:])
print(f"Yeni gorseller dagitildi   -> train={len(new_train)}  val={len(new_val)}  test={len(new_test)}")

# --- (c) birlestir ---
train_image_ids = base_train | new_train
val_image_ids   = base_val   | new_val
test_image_ids  = base_test  | new_test

# --- (d) DOGRULAMALAR ---
assert not (train_image_ids & val_image_ids),  "train/val kesisiyor"
assert not (train_image_ids & test_image_ids), "train/test kesisiyor"
assert not (val_image_ids   & test_image_ids), "val/test kesisiyor"
assert len(train_image_ids | val_image_ids | test_image_ids) == len(_all_ids), "kapsama eksik"
print(f"\n✅ Yeni bolme: train={len(train_image_ids)}  val={len(val_image_ids)}  test={len(test_image_ids)}")

# 2K referans buyuklukleriyle karsilastir (1275 / 225 / 500 bekleniyor)
_expected = (1275, 225, 500)
_got = (len(base_train), len(base_val), len(base_test))
print(f"2K referansi: beklenen {_expected}  -  uretilen {_got}  ->  "
      f"{'EŞLEŞTI' if _got == _expected else 'FARKLI (kontrol et!)'}")

# ESKI (yeniden karistirilmis) bolmeyle farki olc: kac orijinal gorsel kutu degistirdi?
_rng_old = random.Random(42)
_old = _all_ids[:]
_rng_old.shuffle(_old)
_osp = int(len(_old) * TRAIN_RATIO)
_old_train_full, _old_test = set(_old[:_osp]), set(_old[_osp:])
_ort = sorted(_old_train_full); random.Random(43).shuffle(_ort)
_osp2 = int(len(_ort) * (1 - VAL_RATIO))
_old_train, _old_val = set(_ort[:_osp2]), set(_ort[_osp2:])
def _bucket(i, tr, va, te): return "train" if i in tr else ("val" if i in va else "test")
_changed = sum(1 for i in _base_ids
               if _bucket(i, _old_train, _old_val, _old_test) != _bucket(i, train_image_ids, val_image_ids, test_image_ids))
print(f"Eski bolmeye gore kutu degistiren ORIJINAL gorsel: {_changed}/{len(_base_ids)} "
      f"(%{_changed/len(_base_ids)*100:.1f})  <- duzeltilen sapma bu kadardi")

# --- (e) CIKTI YOLLARI: mevcut v5i_02 dosyalarina DOKUNMA, _ps son ekiyle yaz ---
TRAIN_PAIRS_PATH   = f"{DRIVE_BASE}/ZoomInOut/v5i_02_ps_train_pairs.pkl"
VAL_PAIRS_PATH     = f"{DRIVE_BASE}/ZoomInOut/v5i_02_ps_val_pairs.pkl"
TEST_PAIRS_PATH    = f"{DRIVE_BASE}/ZoomInOut/v5i_02_ps_test_pairs.pkl"
CHAIN_RECORDS_PATH = f"{DRIVE_BASE}/ZoomInOut/v5i_02_ps_chain_records.pkl"
# TUM sonuc dosyalari da ayri klasore: eski sonuclarin uzerine YAZILMAZ
SPLIT_PRESERVING = True
RESULTS_DIR = f"{DRIVE_BASE}/ZoomInOut/v5i_02_ps_results"
os.makedirs(RESULTS_DIR, exist_ok=True)
DSACL_FOCUS_CKPT   = f"{RESULTS_DIR}/dsacl_focus_results.pkl"
print("\nCikti yollari '_ps' son ekine alindi:")
for _p in (TRAIN_PAIRS_PATH, VAL_PAIRS_PATH, TEST_PAIRS_PATH, CHAIN_RECORDS_PATH, DSACL_FOCUS_CKPT):
    print("   ", _p.split("/ZoomInOut/")[-1], "  (var)" if os.path.exists(_p) else "  (yok, uretilecek)")
print("\nSIRADAKI ADIM: asagidaki 'HUCRE 7' (train+val+test uretimi) ve sonrasini normal sirayla calistir.")


In [ ]:
# HÜCRE HIZLANDIRMA: HybridMultimodalIndex sorgularini vektorlestir
# ---------------------------------------------------------------------------
# NE YAPAR: query_text / query_image / find_hard_negative_text /
# image_patches_of / text_patches_of / image_ids fonksiyonlarindaki
# "her cagrida 100-250 bin Python nesnesi uzerinde don" davranisini kaldirir.
# Bu dongulerde kullanilan ozellikler (yama tipi, doc_id, pencere boyutu,
# metin kalitesi) STATIKTIR; bir kez onceden hesaplanip numpy dizilerine
# alinir ve sorgular saf vektor islemine doner.
#
# SECIM MANTIGI DEGISMEZ: ayni agirliklar, ayni -999.0 eleme, ayni
# np.argmax / np.argsort cagrilari, ayni donus degerleri. Yalnizca Python
# dongusu kaldirilir. Uretilen sekanslar istatistiksel olarak ayni surecten
# gelir (zaten secrets.choice nedeniyle kosular arasi birebir ayni degildir).
# ---------------------------------------------------------------------------
import numpy as _np, types as _types

def _accelerate_index(ix):
    n_img, n_txt = len(ix.image_patches), len(ix.text_patches)
    print(f"⏳ Indeks hizlandiriliyor: {n_img} gorsel yamasi, {n_txt} metin yamasi...")
    ix._build_matrices()

    TYPE_WEIGHT = {"object": 1.10, "background": 0.92}
    WIN_WEIGHT  = {1: 0.70, 2: 1.05, 3: 1.15}

    # --- gorsel tarafi ---
    _iid_list = sorted({p.image_id for p in ix.image_patches})
    _iid2code = {v: k for k, v in enumerate(_iid_list)}
    ix._acc_iid_list  = _iid_list
    ix._acc_img_code  = _np.fromiter((_iid2code[p.image_id] for p in ix.image_patches),
                                     dtype=_np.int32, count=n_img)
    ix._acc_iid2code  = _iid2code
    ix._acc_img_w     = _np.fromiter((TYPE_WEIGHT.get(p.type, 1.0) for p in ix.image_patches),
                                     dtype=_np.float32, count=n_img)
    _by_iid = {}
    for p in ix.image_patches:
        _by_iid.setdefault(p.image_id, []).append(p)
    ix._acc_by_iid = _by_iid

    # --- metin tarafi ---
    _did_list = sorted({p.doc_id for p in ix.text_patches})
    _did2code = {v: k for k, v in enumerate(_did_list)}
    ix._acc_did2code = _did2code
    ix._acc_txt_code = _np.fromiter((_did2code[p.doc_id] for p in ix.text_patches),
                                    dtype=_np.int32, count=n_txt)
    ix._acc_txt_w    = _np.fromiter((WIN_WEIGHT.get(int(p.win_size), 1.0) for p in ix.text_patches),
                                    dtype=_np.float32, count=n_txt)
    ix._acc_txt_win  = _np.fromiter((int(p.win_size) for p in ix.text_patches),
                                    dtype=_np.int32, count=n_txt)
    ix._acc_txt_ok   = _np.fromiter((bool(ix._is_quality_text(p.text_content)) for p in ix.text_patches),
                                    dtype=bool, count=n_txt)
    _by_did = {}
    for p in ix.text_patches:
        _by_did.setdefault(p.doc_id, []).append(p)
    ix._acc_by_did = _by_did
    ix._acc_n_img, ix._acc_n_txt = n_img, n_txt

    def _img_excl_mask(self, exclude_image_ids):
        m = _np.zeros(len(self._acc_iid_list), dtype=bool)
        if exclude_image_ids:
            c = self._acc_iid2code
            idx = [c[x] for x in exclude_image_ids if x in c]
            if idx:
                m[_np.asarray(idx, dtype=_np.int32)] = True
        return m[self._acc_img_code]

    def _txt_excl_mask(self, exclude_doc_ids):
        m = _np.zeros(len(self._acc_did2code), dtype=bool)
        if exclude_doc_ids:
            c = self._acc_did2code
            idx = [c[x] for x in exclude_doc_ids if x in c]
            if idx:
                m[_np.asarray(idx, dtype=_np.int32)] = True
        return m[self._acc_txt_code]

    def image_ids(self):
        return list(self._acc_iid_list)

    def image_patches_of(self, image_id):
        return self._acc_by_iid.get(image_id, [])

    def text_patches_of(self, doc_id):
        return self._acc_by_did.get(doc_id, [])

    def query_text(self, query_emb, exclude_doc_ids=None, threshold=-1.0, min_win_size=1, top_k=1):
        self._build_matrices()
        if self._txt_matrix is None or not self.text_patches:
            return []
        q = query_emb / (_np.linalg.norm(query_emb) + 1e-9)
        raw = _np.clip(self._txt_matrix @ q, -1.0, 1.0)
        weighted = raw * self._acc_txt_w
        bad = _txt_excl_mask(self, exclude_doc_ids) | (self._acc_txt_win < min_win_size) | (~self._acc_txt_ok)
        weighted = _np.where(bad, -999.0, weighted)
        if top_k <= 1:
            best_idx = int(_np.argmax(weighted))
            if weighted[best_idx] <= -999.0:
                return []
            return [(self.text_patches[best_idx], float(raw[best_idx]))]
        order = _np.argsort(-weighted)[:top_k]
        return [(self.text_patches[i], float(raw[i])) for i in order if weighted[i] > -999.0]

    def query_image(self, query_emb, exclude_image_ids=None, top_k=1):
        self._build_matrices()
        if self._img_matrix is None or not self.image_patches:
            return []
        q = query_emb / (_np.linalg.norm(query_emb) + 1e-9)
        raw = _np.clip(self._img_matrix @ q, -1.0, 1.0)
        weighted = _np.where(_img_excl_mask(self, exclude_image_ids), -999.0, raw * self._acc_img_w)
        if top_k <= 1:
            best_idx = int(_np.argmax(weighted))
            if weighted[best_idx] <= -999.0:
                return []
            return [(self.image_patches[best_idx], float(raw[best_idx]))]
        order = _np.argsort(-weighted)[:top_k]
        return [(self.image_patches[i], float(raw[i])) for i in order if weighted[i] > -999.0]

    def find_hard_negative_text(self, anchor_patch, exclude_doc_ids,
                                 hardness_min=0.05, hard_neg_sem_thresh=0.85):
        self._build_matrices()
        if self._lex_matrix is None or anchor_patch.lexical_feat is None:
            return None
        surface_all = _np.clip(self._lex_matrix @ anchor_patch.lexical_feat, -1.0, 1.0)
        sem_all     = _np.clip(self._txt_matrix @ anchor_patch.embedding, -1.0, 1.0)
        hardness    = surface_all - sem_all
        bad = _txt_excl_mask(self, exclude_doc_ids) | (~self._acc_txt_ok)
        ac = self._acc_did2code.get(anchor_patch.doc_id)
        if ac is not None:
            bad = bad | (self._acc_txt_code == ac)
        h = _np.where(bad, -_np.inf, hardness)
        best_idx = int(_np.argmax(h))
        if not _np.isfinite(h[best_idx]):
            return None
        if h[best_idx] < hardness_min or sem_all[best_idx] >= hard_neg_sem_thresh:
            return None
        return self.text_patches[best_idx].patch_id, float(surface_all[best_idx]), "lexical"

    for _f in (image_ids, image_patches_of, text_patches_of, query_text, query_image, find_hard_negative_text):
        setattr(ix, _f.__name__, _types.MethodType(_f, ix))
    print("✅ Hizlandirma etkin. Secim mantigi degismedi; yalnizca Python donguleri kaldirildi.")

_accelerate_index(hybrid_index)

# --- kisa dogrulama: eski/yeni sonuclar ayni mi + hiz olcumu ---
import time as _t
_q = hybrid_index.text_patches[0].embedding
_t0 = _t.time(); _r = hybrid_index.query_image(_q, exclude_image_ids=set(), top_k=8); _d = _t.time() - _t0
print(f"query_image: {_d*1000:.1f} ms, {len(_r)} aday")
_t0 = _t.time(); _r2 = hybrid_index.query_text(hybrid_index.image_patches[0].embedding,
                                                exclude_doc_ids=set(), min_win_size=2, top_k=8); _d2 = _t.time() - _t0
print(f"query_text : {_d2*1000:.1f} ms, {len(_r2)} aday")
print(f"\\nTahmini sekans suresi: ~{(_d + _d2) * 32:.1f} sn  (onceki: ~23 sn)")


## 6. Değerlendirme kümeleri (ana koşudan aynen yüklenir)

Val ve test kümeleri **yeniden üretilmez**; split-koruyan ana koşuda üretilen
`_ps` dosyalarından yüklenir. Böylece bu deneydeki tek değişken eğitim
verisinin hacmi olur.

In [ ]:
# HÜCRE VT: val/test ciftlerini ana kosudan yukle (yeniden URETME)
for _p in (VAL_PAIRS_PATH, TEST_PAIRS_PATH):
    assert os.path.exists(_p), (f"EKSIK: {_p}\nAna kosu (ZoomInOut_v5i_02_dataset.ipynb, "
                                 "PS1 + HUCRE 7) once calistirilmis olmali.")
with open(VAL_PAIRS_PATH, "rb") as f:
    _d = pickle.load(f); val_pairs = _d["val_pairs"] if isinstance(_d, dict) and "val_pairs" in _d else _d
with open(TEST_PAIRS_PATH, "rb") as f:
    _d = pickle.load(f); test_pairs = _d["test_pairs"] if isinstance(_d, dict) and "test_pairs" in _d else _d
print(f"val_pairs : {len(val_pairs)}")
print(f"test_pairs: {len(test_pairs)}")
print("\nBu kumeler 600-sekans kosusuyla AYNI -- karsilastirma kontrollu.")

## 7. Sekans üretimi (CPU yeterli, kesintiye dayanıklı)

In [ ]:
# HÜCRE SEQ1 (KESINTIYE DAYANIKLI): sekans sayisini artirarak train verisini uret
# Uretim parcalar halinde yapilir ve her parca sonunda ilerleme kaydedilir.
# Kesilirse ayni hucreyi tekrar calistir: kaldigi sekanstan devam eder.
# val/test DEGISMEZ; yalnizca egitim verisi buyur.
for _need in ["hybrid_index","train_image_ids","val_pairs","test_pairs","MODE_CALIB",
              "build_pairs_from_sequences","attach_embeddings"]:
    assert _need in globals(), f"ONKOSUL EKSIK: {_need} bellekte yok. Notebook'u sirayla calistir."
if not globals().get("SPLIT_PRESERVING", False):
    print("⚠️ UYARI: PS1 calismamis gorunuyor. Bu deney split-koruyan bolme uzerinde yapilmalidir.")

try:
    import psutil
    def _ram(tag=""):
        v = psutil.virtual_memory()
        print(f"   [RAM] {tag:<24} kullanilan {v.used/2**30:.1f} / {v.total/2**30:.1f} GB")
except Exception:
    def _ram(tag=""): pass

# ---------------- ayarlar ----------------
NUM_SEQUENCES_BIG = 800     # dosya adlari bu degerden turetilir; degistirirsen ayri kosu olur
CHUNK             = 50      # kac sekansta bir ilerleme kaydedilsin
SEQ_TAG              = f"seq{NUM_SEQUENCES_BIG}"
SEQ_TRAIN_PAIRS_PATH = f"{DRIVE_BASE}/ZoomInOut/v5i_02_ps_{SEQ_TAG}_train_pairs.pkl"
SEQ_CHAIN_PATH       = f"{DRIVE_BASE}/ZoomInOut/v5i_02_ps_{SEQ_TAG}_chain_records.pkl"
SEQ_RESULTS_DIR      = f"{DRIVE_BASE}/ZoomInOut/v5i_02_ps_{SEQ_TAG}_results"
SEQ_PROGRESS_PATH    = f"{DRIVE_BASE}/ZoomInOut/v5i_02_ps_{SEQ_TAG}_progress.pkl"
os.makedirs(SEQ_RESULTS_DIR, exist_ok=True)

# ---------------- eski veriyi bellekten dusur ----------------
FREE_OLD_DATA = True
_ram("baslangic")
if FREE_OLD_DATA:
    for _v in ["train_pairs","train_chain_records","train_loader","chain_loader",
               "train_sequences","random_pairs","random_loader"]:
        globals().pop(_v, None)
    gc.collect(); _ram("eski veri bosaltildi")

print(f"Train havuzu: {len(train_image_ids)} gorsel")
print(f"Sekans: 600 (onceki) -> {NUM_SEQUENCES_BIG} (bu deney) | tam orantili: "
      f"{round(600*len(train_image_ids)/1275)}")
print(f"Gorsel basina: {600/len(train_image_ids):.3f} -> {NUM_SEQUENCES_BIG/len(train_image_ids):.3f}\n")

# ---------------- nihai cikti zaten varsa atla ----------------
if os.path.exists(SEQ_TRAIN_PAIRS_PATH) and os.path.exists(SEQ_CHAIN_PATH):
    with open(SEQ_TRAIN_PAIRS_PATH, "rb") as f: _d = pickle.load(f)
    seq_train_pairs = _d["train_pairs"]; _cov = _d.get("coverage_pct")
    with open(SEQ_CHAIN_PATH, "rb") as f: seq_chain_records = pickle.load(f)
    print(f"✅ Nihai cikti zaten var: {len(seq_train_pairs)} cift, {len(seq_chain_records)} zincir"
          + (f", kapsam %{_cov:.1f}" if _cov else ""))
else:
    # ---------------- baslangic sirasi: OTURUMLAR ARASI AYNI ----------------
    # sorted() sart: set sirasi Python surecleri arasinda degisebiliyor,
    # devam edebilmek icin sira sabit olmali.
    _starts = sorted(train_image_ids)
    random.Random(1234).shuffle(_starts)

    # ---------------- ilerleme dosyasi ----------------
    if os.path.exists(SEQ_PROGRESS_PATH):
        with open(SEQ_PROGRESS_PATH, "rb") as f: _prog = pickle.load(f)
        print(f"↻ Ilerleme bulundu: {_prog['done']}/{NUM_SEQUENCES_BIG} sekans tamam, devam ediliyor.\n")
    else:
        _prog = {"done": 0, "records": [], "chains_light": [], "kept": 0,
                 "image_use_count": {}, "doc_use_count": {}}

    seq_engine = MultimodalSpiralEngine(index=hybrid_index, similarity_threshold=SIM_THRESHOLD,
                                         max_iou_shift=0.1, min_win_size=2, verbose=False,
                                         hardness_min=HARDNESS_MIN, hard_neg_sem_thresh=HARD_NEG_SEM_THRESH,
                                         mode_calib=MODE_CALIB, allowed_image_ids=train_image_ids,
                                         top_k=TOP_K, diversity_alpha=DIVERSITY_ALPHA)
    # kullanim sayaclarini geri yukle -- hub duzeltmesinin surekliligi icin sart
    seq_engine.image_use_count = dict(_prog["image_use_count"])
    seq_engine.doc_use_count   = dict(_prog["doc_use_count"])

    def _save_progress():
        _prog["image_use_count"] = dict(seq_engine.image_use_count)
        _prog["doc_use_count"]   = dict(seq_engine.doc_use_count)
        _tmp = SEQ_PROGRESS_PATH + ".tmp"
        with open(_tmp, "wb") as f: pickle.dump(_prog, f)
        os.replace(_tmp, SEQ_PROGRESS_PATH)   # yarim dosya birakmaz

    _t0 = time.time()
    _pbar = tqdm(range(_prog["done"], NUM_SEQUENCES_BIG), initial=_prog["done"],
                  total=NUM_SEQUENCES_BIG, desc="Sekans")
    _buf = []
    for _i in _pbar:
        _chain = seq_engine.run_hybrid_loop(start_image_id=_starts[_i % len(_starts)],
                                             loops=LOOPS_PER_SEQUENCE)
        if len(_chain) >= MIN_CHAIN_LENGTH:
            _buf.append(_chain); _prog["kept"] += 1
        _prog["done"] = _i + 1

        if len(_buf) >= CHUNK or _prog["done"] == NUM_SEQUENCES_BIG:
            if _buf:
                # agir embedding'ler DEGIL, hafif kimlikler saklanir
                _prog["records"].extend(build_pairs_from_sequences(_buf, SIM_THRESHOLD))
                for _c in _buf:
                    _prog["chains_light"].append(
                        [(s.step, 0 if s.modality == "image" else 1, s.patch_id) for s in _c])
                _buf = []
            _save_progress()
            _pbar.set_postfix(kept=_prog["kept"], kayit=len(_prog["records"]))

    print(f"\n✅ Uretim tamam: {_prog['kept']}/{NUM_SEQUENCES_BIG} sekans filtreden gecti "
          f"({(time.time()-_t0)/60:.1f} dk)")
    _touched = len(set(seq_engine.image_use_count.keys()))
    _cov = _touched / len(train_image_ids) * 100
    print(f"[KAPSAM] motor {_touched}/{len(train_image_ids)} farkli gorsele dokundu (%{_cov:.1f})")
    _ram("uretim sonu")

    # ---------------- embedding'leri en sonda materyalize et ----------------
    seq_train_pairs = attach_embeddings(_prog["records"], hybrid_index)
    with open(SEQ_TRAIN_PAIRS_PATH, "wb") as f:
        pickle.dump({"train_pairs": seq_train_pairs, "coverage_pct": _cov,
                     "config": {"num_sequences": NUM_SEQUENCES_BIG,
                                "loops_per_sequence": LOOPS_PER_SEQUENCE,
                                "sim_threshold": SIM_THRESHOLD, "top_k": TOP_K,
                                "diversity_alpha": DIVERSITY_ALPHA, "split": "preserve-2K"}}, f)

    _img_lut = {p.patch_id: p.embedding for p in hybrid_index.image_patches}
    _txt_lut = {p.patch_id: p.embedding for p in hybrid_index.text_patches}
    seq_chain_records = []
    for _cl in _prog["chains_light"]:
        _nodes = []
        for _st, _mod, _pid in _cl:
            _e = _img_lut.get(_pid) if _mod == 0 else _txt_lut.get(_pid)
            if _e is not None:
                _nodes.append({"step": _st, "modality": _mod,
                               "embedding": np.asarray(_e, dtype=np.float32)})
        if len(_nodes) >= 8:
            seq_chain_records.append(_nodes)
    with open(SEQ_CHAIN_PATH, "wb") as f: pickle.dump(seq_chain_records, f)
    del _img_lut, _txt_lut, _prog, seq_engine
    gc.collect(); _ram("materyalizasyon sonu")
    print(f"✅ Kaydedildi: {len(seq_train_pairs)} cift, {len(seq_chain_records)} zincir")
    if os.path.exists(SEQ_PROGRESS_PATH):
        os.remove(SEQ_PROGRESS_PATH)   # is bitti, ilerleme dosyasi gereksiz

_prev_n = None
if os.path.exists(TRAIN_PAIRS_PATH):
    with open(TRAIN_PAIRS_PATH, "rb") as f: _prev_n = len(pickle.load(f)["train_pairs"])
print(f"\nOnceki kosu (600 sekans): {_prev_n or '?'} cift  |  bu kosu: {len(seq_train_pairs)} cift")


## 8. Model, kayıp ve eğitim altyapısı (GPU buradan itibaren)

In [ ]:
# HÜCRE 10: Model mimarileri (v5h: residual/adapter mimari)
# Eskiden projeksiyon katmani ham embedding'i SIFIRDAN yeniden ogreniyordu
# (rastgele baslatilmis agirliklarla) VE 2048->512 boyut daraltmasi yapiyordu
# -- bu, kucuk/gurultulu bir egitim sinyaliyle zaten guclu olan ham Qwen
# temsilini bozma riski tasiyordu (nitekim bazi varyantlar ham modeli
# gecemiyordu). Simdi LoRA/adapter mantigiyla: agirliklar ham embedding'e
# KUCUK bir duzeltme (delta) ogreniyor, son katman SIFIR baslatiliyor --
# yani egitimin basinda delta=0 ve cikti TAM OLARAK ham (normalize edilmis)
# embedding'e esit. Egitim SADECE iyilestirme yonunde ilerleyebilir, ham
# modelden daha kotu bir noktadan baslamiyor.

EMBED_DIM = 2048
HIDDEN    = 1024
TEMP      = 0.07


class ResidualHead(nn.Module):
    """Ham embedding + kucuk ogrenilebilir duzeltme (delta). Son katman
    sifir-baslatilir -- baslangicta delta=0, cikti = normalize(ham_embedding)."""
    def __init__(self, embed_dim=EMBED_DIM, context_dim=0, hidden=HIDDEN):
        super().__init__()
        self.mlp = nn.Sequential(nn.Linear(embed_dim + context_dim, hidden), nn.GELU(),
                                  nn.LayerNorm(hidden), nn.Linear(hidden, embed_dim))
        nn.init.zeros_(self.mlp[-1].weight)
        nn.init.zeros_(self.mlp[-1].bias)

    def forward(self, x, context=None):
        inp = x if context is None else torch.cat([x, context], dim=-1)
        return F.normalize(x + self.mlp(inp), dim=-1)


class ProjectionHead(nn.Module):
    """A) Baseline / B) Hard-negative-aware / E) SACL için ortak mimari."""
    def __init__(self, input_dim=EMBED_DIM, hidden=HIDDEN):
        super().__init__()
        self.img_proj = ResidualHead(input_dim, context_dim=0, hidden=hidden)
        self.txt_proj = ResidualHead(input_dim, context_dim=0, hidden=hidden)

    def forward(self, img, txt):
        return self.img_proj(img), self.txt_proj(txt)

    def project_image(self, img):
        return self.img_proj(img)


class InstructionAwareHead(nn.Module):
    """C) Instruction-aware (ORACLE mod gerektirir -- sadece üst-sınır/referans)."""
    def __init__(self, input_dim=EMBED_DIM, hidden=HIDDEN, num_modes=4):
        super().__init__()
        self.mode_emb = nn.Embedding(num_modes, input_dim)
        self.img_proj = ResidualHead(input_dim, context_dim=input_dim, hidden=hidden)
        self.txt_proj = ResidualHead(input_dim, context_dim=input_dim, hidden=hidden)

    def forward(self, img, txt, mode_idx):
        m = self.mode_emb(mode_idx)
        return self.img_proj(img, m), self.txt_proj(txt, m)

    def project_image(self, img, mode_idx):
        m = self.mode_emb(mode_idx)
        return self.img_proj(img, m)


class SelfInferringHead(nn.Module):
    """D) Self-inferring (ORACLE YOK) -- gerçek kullanım için asıl aday."""
    def __init__(self, input_dim=EMBED_DIM, hidden=HIDDEN, num_modes=4):
        super().__init__()
        self.mode_table = nn.Embedding(num_modes, input_dim)
        self.mode_predictor = nn.Sequential(nn.Linear(input_dim * 2, 256), nn.GELU(), nn.Linear(256, num_modes))
        self.img_proj = ResidualHead(input_dim, context_dim=input_dim, hidden=hidden)
        self.txt_proj = ResidualHead(input_dim, context_dim=input_dim, hidden=hidden)

    def infer_mode(self, img, txt):
        logits = self.mode_predictor(torch.cat([img, txt], dim=-1))
        probs = F.softmax(logits, dim=-1)
        mode_vec = probs @ self.mode_table.weight
        return mode_vec, logits

    def forward(self, img, txt):
        mode_vec, mode_logits = self.infer_mode(img, txt)
        z_img = self.img_proj(img, mode_vec)
        z_txt = self.txt_proj(txt, mode_vec)
        return z_img, z_txt, mode_logits

    def project_image_with_vec(self, img, mode_vec):
        return self.img_proj(img, mode_vec)


In [ ]:
# HÜCRE 11: Loss fonksiyonları

def plain_info_nce(img, txt, **kwargs):
    batch_size = img.size(0)
    logits = img @ txt.T / TEMP
    labels = torch.arange(batch_size, device=img.device)
    return (F.cross_entropy(logits, labels) + F.cross_entropy(logits.T, labels)) / 2


def hn_info_nce(img, txt, z_hn, has_hn, **kwargs):
    batch_size = img.size(0)
    labels = torch.arange(batch_size, device=img.device)
    logits_i2t = img @ txt.T / TEMP
    loss_i2t = F.cross_entropy(logits_i2t, labels)
    logits_t2i_base = txt @ img.T / TEMP
    # DUZELTME: batch'te HIC hard-negative yoksa z_hn=None gelir -- eskiden
    # bu durumda dogrudan (txt * None) carpimina girip TypeError firlatiyordu.
    # Simdi z_hn None ise o ek sutunu atlayip normal InfoNCE'ye dusuyor.
    if z_hn is not None:
        sim_txt_hn = (txt * z_hn).sum(-1, keepdim=True) / TEMP
        sim_txt_hn = sim_txt_hn.masked_fill((has_hn.unsqueeze(1) == 0), float("-inf"))
        logits_t2i = torch.cat([logits_t2i_base, sim_txt_hn], dim=1)
    else:
        logits_t2i = logits_t2i_base
    loss_t2i = F.cross_entropy(logits_t2i, labels)
    return (loss_i2t + loss_t2i) / 2


def hn_info_nce_realneg(img, txt, z_hn, has_hn, z_chn=None, has_chn=None, **kwargs):
    """v5h: hn_info_nce'nin genisletilmisi -- CV-tabanli hard-negative'e (z_hn)
    EK olarak, gercek COCO kategori eslesmesiyle bulunmus semantik
    hard-negative'i (z_chn, varsa) de ayri bir sutun olarak ekliyor.
    z_chn/has_chn verilmezse davranis hn_info_nce ile BIREBIR AYNI."""
    batch_size = img.size(0)
    labels = torch.arange(batch_size, device=img.device)
    logits_i2t = img @ txt.T / TEMP
    loss_i2t = F.cross_entropy(logits_i2t, labels)
    logits_t2i_base = txt @ img.T / TEMP
    extra_cols = [logits_t2i_base]
    if z_hn is not None:
        sim_txt_hn = (txt * z_hn).sum(-1, keepdim=True) / TEMP
        sim_txt_hn = sim_txt_hn.masked_fill((has_hn.unsqueeze(1) == 0), float("-inf"))
        extra_cols.append(sim_txt_hn)
    if z_chn is not None and has_chn is not None:
        sim_txt_chn = (txt * z_chn).sum(-1, keepdim=True) / TEMP
        sim_txt_chn = sim_txt_chn.masked_fill((has_chn.unsqueeze(1) == 0), float("-inf"))
        extra_cols.append(sim_txt_chn)
    logits_t2i = torch.cat(extra_cols, dim=1)
    loss_t2i = F.cross_entropy(logits_t2i, labels)
    return (loss_i2t + loss_t2i) / 2


MODE_MAP = {"semantic": 0, "color": 1, "shape": 2, "texture": 3}
MODE_AUX_WEIGHT = 0.1  # tarama sonucu: 0.1 en iyi MRR (0.363), >=2.0 mod_aux baskinlasip R@1 dusuruyor


In [ ]:
# HÜCRE 12: Dataset + eğitim/değerlendirme döngüleri

BATCH_SIZE = 128
LR = 1e-4


class PairDataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        p = self.pairs[idx]
        img = torch.tensor(np.asarray(p["img_embedding"], dtype=np.float32))
        txt = torch.tensor(np.asarray(p["txt_embedding"], dtype=np.float32))
        hn = p.get("hard_negative_embedding")
        has_hn = hn is not None
        hn_t = torch.tensor(np.asarray(hn, dtype=np.float32)) if has_hn else torch.zeros_like(img)
        # Fikir 2: metin-tarafı hard-negative (varsa) -- yeniden üretilen veri kümesinde bulunur
        hn_txt = p.get("hard_negative_txt_embedding")
        has_hn_txt = hn_txt is not None
        hn_txt_t = torch.tensor(np.asarray(hn_txt, dtype=np.float32)) if has_hn_txt else torch.zeros_like(txt)
        mode = p.get("hard_negative_mode") or "semantic"
        mode_idx = MODE_MAP.get(mode, 0)
        # v5h: gercek COCO-kategori hard-negative (varsa) -- opsiyonel, geriye
        # donuk uyumlu (yoksa has_chn=0, mevcut yontemleri hic etkilemez).
        chn = p.get("category_hard_negative_embedding")
        has_chn = chn is not None
        chn_t = torch.tensor(np.asarray(chn, dtype=np.float32)) if has_chn else torch.zeros_like(img)
        return {"img": img, "txt": txt, "hn": hn_t,
                "has_hn": torch.tensor(1.0 if has_hn else 0.0),
                "hn_txt": hn_txt_t,
                "has_hn_txt": torch.tensor(1.0 if has_hn_txt else 0.0),
                "chn": chn_t,
                "has_chn": torch.tensor(1.0 if has_chn else 0.0),
                "mode_idx": torch.tensor(mode_idx, dtype=torch.long)}


# train_loader SEQ2 hucresinde YENI veriyle kurulur
if "train_pairs" in globals():
    train_loader = DataLoader(PairDataset(train_pairs), batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
else:
    print("train_pairs bellekte yok -- train_loader SEQ2 hucresinde kurulacak.")


def train_model(model_type, loss_fn_name, epochs, val_pairs=None, patience=4, eval_every=5, loader=None, seed=None):
    """v5h: cosine LR schedule + val-tabanli erken durdurma + en-iyi-checkpoint.
    val_pairs verilirse, her eval_every epoch'ta bir val R@1 olculur; en iyi
    R@1'i veren agirliklar saklanir (deep copy) ve egitim sonunda GERI
    YUKLENIR -- yani donen model, son epoch'un degil, val'e gore EN IYI
    epoch'un modelidir. patience kontrol turunda iyilesme olmazsa erken durur.
    v5h-fix: 'seed' parametresi eklendi -- eskiden hep global SEED sabitine
    resetleniyordu, cok-seed dogrulamasindaki gercek seed varyasyonunu
    SESSIZCE eziyordu (A/B/C her seed'de birebir ayni sonucu veriyordu)."""
    torch.manual_seed(seed if seed is not None else SEED)
    if model_type == "plain":
        model = ProjectionHead().to(device)
    elif model_type == "instruction":
        model = InstructionAwareHead().to(device)
    else:
        model = SelfInferringHead().to(device)

    loss_fn = plain_info_nce if loss_fn_name == "plain" else hn_info_nce
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    best_state = copy.deepcopy(model.state_dict())
    best_val_r1 = -1.0
    checks_since_improve = 0

    data_loader = loader if loader is not None else train_loader

    for epoch in range(epochs):
        model.train()
        epoch_loss, epoch_aux = 0.0, 0.0
        for batch in data_loader:
            img = batch["img"].to(device); txt = batch["txt"].to(device)
            hn = batch["hn"].to(device); has_hn = batch["has_hn"].to(device)
            mode_idx = batch["mode_idx"].to(device)

            optimizer.zero_grad()
            if model_type == "plain":
                z_img, z_txt = model(img, txt)
                z_hn = model.project_image(hn) if loss_fn_name == "hn" else None
                loss = loss_fn(z_img, z_txt, z_hn=z_hn, has_hn=has_hn)
                aux = torch.tensor(0.0)
            elif model_type == "instruction":
                z_img, z_txt = model(img, txt, mode_idx)
                z_hn = model.project_image(hn, mode_idx) if loss_fn_name == "hn" else None
                loss = loss_fn(z_img, z_txt, z_hn=z_hn, has_hn=has_hn)
                aux = torch.tensor(0.0)
            else:
                z_img, z_txt, mode_logits = model(img, txt)
                z_hn = None
                if loss_fn_name == "hn":
                    mode_vec_hn, _ = model.infer_mode(img, txt)
                    z_hn = model.project_image_with_vec(hn, mode_vec_hn)
                base_loss = loss_fn(z_img, z_txt, z_hn=z_hn, has_hn=has_hn)
                mask = has_hn.bool()
                aux = F.cross_entropy(mode_logits[mask], mode_idx[mask]) if mask.sum() > 0 else torch.tensor(0.0)
                loss = base_loss + MODE_AUX_WEIGHT * aux

            loss.backward()
            optimizer.step()
            epoch_loss += loss.item(); epoch_aux += aux.item()
        scheduler.step()

        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"  [{model_type}/{loss_fn_name}] epoch {epoch+1}/{epochs} loss={epoch_loss/len(data_loader):.4f} mode_aux={epoch_aux/len(data_loader):.4f} lr={scheduler.get_last_lr()[0]:.2e}")

        if val_pairs is not None and ((epoch + 1) % eval_every == 0 or epoch + 1 == epochs):
            r_val = evaluate(model, model_type, val_pairs, forced_mode=None)
            if r_val["R@1"] > best_val_r1:
                best_val_r1 = r_val["R@1"]
                best_state = copy.deepcopy(model.state_dict())
                checks_since_improve = 0
            else:
                checks_since_improve += 1
            if checks_since_improve >= patience:
                print(f"  [erken durdurma] val R@1 {patience} kontroldur iyilesmedi (en iyi %{best_val_r1:.1f}), epoch {epoch+1}'de durduruluyor.")
                break

    if val_pairs is not None:
        model.load_state_dict(best_state)
        print(f"  [{model_type}/{loss_fn_name}] en iyi val R@1={best_val_r1:.1f}% olan checkpoint geri yuklendi.")
    return model


@torch.no_grad()
def evaluate(model, model_type, pairs, forced_mode=None):
    model.eval()
    img_all = torch.tensor(np.stack([np.asarray(p["img_embedding"], dtype=np.float32) for p in pairs])).to(device)
    txt_all = torch.tensor(np.stack([np.asarray(p["txt_embedding"], dtype=np.float32) for p in pairs])).to(device)

    if model_type == "plain":
        z_img, z_txt = model(img_all, txt_all)
    elif model_type == "instruction":
        if forced_mode is not None:
            mode_idx = torch.full((len(pairs),), MODE_MAP[forced_mode], dtype=torch.long, device=device)
        else:
            mode_idx = torch.tensor([MODE_MAP.get(p.get("hard_negative_mode") or "semantic", 0) for p in pairs], dtype=torch.long, device=device)
        z_img, z_txt = model(img_all, txt_all, mode_idx)
    else:
        z_img, z_txt, mode_logits = model(img_all, txt_all)
        pred_mode = mode_logits.argmax(dim=-1)
        true_mode = torch.tensor([MODE_MAP.get(p.get("hard_negative_mode") or "semantic", 0) for p in pairs], dtype=torch.long, device=device)
        acc = (pred_mode == true_mode).float().mean().item()
        print(f"  [self_infer] mod tahmin doğruluğu (bilgi amaçlı, oracle DEĞİL): %{acc*100:.1f}")

    sim = (z_txt @ z_img.T).cpu().numpy()
    n = len(pairs)
    r1 = r5 = r10 = 0
    mrr = 0.0
    for i in range(n):
        order = np.argsort(-sim[i])
        rank = int(np.where(order == i)[0][0]) + 1
        if rank == 1: r1 += 1
        if rank <= 5: r5 += 1
        if rank <= 10: r10 += 1
        mrr += 1.0 / rank
    return {"R@1": r1/n*100, "R@5": r5/n*100, "R@10": r10/n*100, "MRR": mrr/n}


In [ ]:
# HÜCRE 14: SACL çekirdek sınıf/fonksiyonlar

class ChainBatch:
    def __init__(self, emb, modality, mask, step_idx):
        self.emb = emb
        self.modality = modality
        self.mask = mask
        self.step_idx = step_idx

    def to(self, device):
        return ChainBatch(self.emb.to(device), self.modality.to(device),
                           self.mask.to(device), self.step_idx.to(device))


def project_chain_batch(model, batch: "ChainBatch"):
    B, L, D = batch.emb.shape
    flat = batch.emb.view(B * L, D)
    z_img = F.normalize(model.img_proj(flat), dim=-1)
    z_txt = F.normalize(model.txt_proj(flat), dim=-1)
    mod = batch.modality.view(B * L, 1).float()
    z = z_img * (1 - mod) + z_txt * mod
    return z.view(B, L, -1)


def sequence_aware_loss(z, batch: "ChainBatch", temp=0.07, pos_window=4,
                         lambda_multipos=0.5, lambda_rank=0.5, margin=0.1,
                         max_rank_samples_per_chain=6, same_modality_only=False):
    """same_modality_only=False: ORİJİNAL SACL (v1) -- multipos/rank modaliteyi
    hesaba katmıyor, bir görsel düğümle pencere-içi bir metin düğümünü de
    "pozitif" sayıyor. LAMBDA_SEQ taramasında bu, en iyi ayarda bile
    A_baseline'ı geçemedi (bkz. sweep hücresi) -- muhtemel sebep: l_align zaten
    çapraz-modal eşleşmeyi net biçimde öğretiyor, multipos/rank'in modaliteyi
    görmezden gelmesi ona çelişen/gürültülü bir sinyal ekliyor.

    same_modality_only=True (SACL v2): multipos ve rank SADECE aynı modalitedeki
    düğümler arasında hesaplanır (görsel<->görsel, metin<->metin) -- "aynı
    içeriğin farklı zoom seviyeleri kümelenmeli" fikri, l_align'ın çapraz-modal
    hedefiyle çakışmadan, ona ek/tamamlayıcı bir intra-modal yapı öğretir.
    """
    B, L, Dp = z.shape
    device = z.device

    flat_z, flat_chain, flat_step, flat_mod = [], [], [], []
    for b in range(B):
        valid = batch.mask[b]
        n_valid = int(valid.sum().item())
        if n_valid == 0:
            continue
        flat_z.append(z[b, valid])
        flat_chain.append(torch.full((n_valid,), b, device=device, dtype=torch.long))
        flat_step.append(batch.step_idx[b, valid])
        flat_mod.append(batch.modality[b, valid])
    flat_z = torch.cat(flat_z, dim=0)
    flat_chain = torch.cat(flat_chain, dim=0)
    flat_step = torch.cat(flat_step, dim=0)
    flat_mod = torch.cat(flat_mod, dim=0)
    N = flat_z.size(0)

    sim = flat_z @ flat_z.T / temp
    same_chain = flat_chain.unsqueeze(0) == flat_chain.unsqueeze(1)
    same_modality = flat_mod.unsqueeze(0) == flat_mod.unsqueeze(1)
    step_dist = (flat_step.unsqueeze(0) - flat_step.unsqueeze(1)).abs()
    is_self = torch.eye(N, dtype=torch.bool, device=device)

    pos_mask = same_chain & (step_dist > 0) & (step_dist <= pos_window) & (~is_self)
    if same_modality_only:
        pos_mask = pos_mask & same_modality
    logits = sim.masked_fill(is_self, float("-inf"))
    log_prob = logits - torch.logsumexp(logits, dim=1, keepdim=True)
    log_prob = log_prob.masked_fill(is_self, 0.0)
    pos_counts = pos_mask.sum(dim=1).clamp(min=1)
    per_anchor = -(log_prob * pos_mask).sum(dim=1) / pos_counts
    has_pos = pos_mask.sum(dim=1) > 0
    l_multipos = per_anchor[has_pos].mean() if has_pos.any() else torch.zeros((), device=device)

    rank_terms = []
    for b in range(B):
        idx_b = (flat_chain == b).nonzero(as_tuple=True)[0]
        if idx_b.numel() < 3:
            continue
        n_sample = min(max_rank_samples_per_chain, idx_b.numel())
        perm = idx_b[torch.randperm(idx_b.numel(), device=device)[:n_sample]]
        for a in perm:
            others = idx_b[idx_b != a]
            if same_modality_only:
                others = others[flat_mod[others] == flat_mod[a]]
            if others.numel() < 2:
                continue
            d = (flat_step[others] - flat_step[a]).abs()
            order = torch.argsort(d)
            near, far = others[order[0]], others[order[-1]]
            if d[order[-1]] <= d[order[0]]:
                continue
            s_near = (flat_z[a] * flat_z[near]).sum() / temp
            s_far = (flat_z[a] * flat_z[far]).sum() / temp
            rank_terms.append(F.relu(torch.tensor(margin, device=device) - (s_near - s_far)))
    l_rank = torch.stack(rank_terms).mean() if rank_terms else torch.zeros((), device=device)

    return lambda_multipos * l_multipos + lambda_rank * l_rank, {
        "l_multipos": l_multipos.item(), "l_rank": l_rank.item(),
        "n_pos_anchors": int(has_pos.sum().item()), "n_rank_triplets": len(rank_terms),
    }


def sequence_aware_loss_continuous(z, batch: "ChainBatch", temp=0.07, tau=3.0,
                                    lambda_multipos=0.5, lambda_rank=0.5, margin=0.1,
                                    max_rank_samples_per_chain=6, same_modality_only=True,
                                    max_weight_dist=None):
    """v5h: pos_window'daki SERT esik yerine, adim mesafesine gore YUMUSAK
    (exponentially decaying) agirlik kullanan surekli SACL varyanti --
    weight(i,j) = exp(-step_dist(i,j)/tau). Zincirdeki TUM ciftler
    kullanilir, yakin adimlar yuksek, uzak adimlar dusuk agirlikla katkida
    bulunur -- ikili (pozitif/degil) ayrim yerine surekli bir "kavramsal
    mesafe" sinyali. Rank (margin) kismi degismedi, sadece multipos kismi
    surekli agirliklandirildi."""
    B, L, Dp = z.shape
    device = z.device

    flat_z, flat_chain, flat_step, flat_mod = [], [], [], []
    for b in range(B):
        valid = batch.mask[b]
        n_valid = int(valid.sum().item())
        if n_valid == 0:
            continue
        flat_z.append(z[b, valid])
        flat_chain.append(torch.full((n_valid,), b, device=device, dtype=torch.long))
        flat_step.append(batch.step_idx[b, valid])
        flat_mod.append(batch.modality[b, valid])
    flat_z = torch.cat(flat_z, dim=0)
    flat_chain = torch.cat(flat_chain, dim=0)
    flat_step = torch.cat(flat_step, dim=0)
    flat_mod = torch.cat(flat_mod, dim=0)
    N = flat_z.size(0)

    sim = flat_z @ flat_z.T / temp
    same_chain = flat_chain.unsqueeze(0) == flat_chain.unsqueeze(1)
    same_modality = flat_mod.unsqueeze(0) == flat_mod.unsqueeze(1)
    step_dist = (flat_step.unsqueeze(0) - flat_step.unsqueeze(1)).abs().float()
    is_self = torch.eye(N, dtype=torch.bool, device=device)

    eligible = same_chain & (~is_self)
    if same_modality_only:
        eligible = eligible & same_modality
    weight = torch.exp(-step_dist / tau) * eligible.float()
    if max_weight_dist is not None:
        weight = weight * (step_dist <= max_weight_dist).float()

    logits = sim.masked_fill(is_self, float("-inf"))
    log_prob = logits - torch.logsumexp(logits, dim=1, keepdim=True)
    log_prob = log_prob.masked_fill(is_self, 0.0)

    weight_sum = weight.sum(dim=1).clamp(min=1e-8)
    per_anchor = -(log_prob * weight).sum(dim=1) / weight_sum
    has_pos = weight.sum(dim=1) > 1e-6
    l_multipos = per_anchor[has_pos].mean() if has_pos.any() else torch.zeros((), device=device)

    rank_terms = []
    for b in range(B):
        idx_b = (flat_chain == b).nonzero(as_tuple=True)[0]
        if idx_b.numel() < 3:
            continue
        n_sample = min(max_rank_samples_per_chain, idx_b.numel())
        perm = idx_b[torch.randperm(idx_b.numel(), device=device)[:n_sample]]
        for a in perm:
            others = idx_b[idx_b != a]
            if same_modality_only:
                others = others[flat_mod[others] == flat_mod[a]]
            if others.numel() < 2:
                continue
            d = (flat_step[others] - flat_step[a]).abs()
            order = torch.argsort(d)
            near, far = others[order[0]], others[order[-1]]
            if d[order[-1]] <= d[order[0]]:
                continue
            s_near = (flat_z[a] * flat_z[near]).sum() / temp
            s_far = (flat_z[a] * flat_z[far]).sum() / temp
            rank_terms.append(F.relu(torch.tensor(margin, device=device) - (s_near - s_far)))
    l_rank = torch.stack(rank_terms).mean() if rank_terms else torch.zeros((), device=device)

    return lambda_multipos * l_multipos + lambda_rank * l_rank, {
        "l_multipos": l_multipos.item(), "l_rank": l_rank.item(),
        "n_pos_anchors": int(has_pos.sum().item()), "n_rank_triplets": len(rank_terms),
    }


In [ ]:
class ChainDataset(Dataset):
    def __init__(self, chain_records, max_len=65, embed_dim=EMBED_DIM):
        self.chains = chain_records
        self.max_len = max_len
        self.embed_dim = embed_dim

    def __len__(self):
        return len(self.chains)

    def __getitem__(self, idx):
        nodes = self.chains[idx][:self.max_len]
        emb = torch.zeros(self.max_len, self.embed_dim)
        modality = torch.zeros(self.max_len, dtype=torch.long)
        mask = torch.zeros(self.max_len, dtype=torch.bool)
        step_idx = torch.zeros(self.max_len, dtype=torch.long)
        for i, n in enumerate(nodes):
            emb[i] = torch.tensor(n["embedding"]); modality[i] = n["modality"]
            step_idx[i] = n["step"]; mask[i] = True
        return emb, modality, mask, step_idx


def chain_collate(batch_list):
    emb = torch.stack([b[0] for b in batch_list])
    modality = torch.stack([b[1] for b in batch_list])
    mask = torch.stack([b[2] for b in batch_list])
    step_idx = torch.stack([b[3] for b in batch_list])
    return ChainBatch(emb, modality, mask, step_idx)


CHAIN_BATCH_SIZE = 16
POS_WINDOW = 4
LAMBDA_MULTIPOS = 0.5
LAMBDA_RANK = 0.5
RANK_MARGIN = 0.1
SACL_MAX_LEN = 65

if "train_chain_records" in globals():
    chain_loader = DataLoader(ChainDataset(train_chain_records, max_len=SACL_MAX_LEN),
                               batch_size=CHAIN_BATCH_SIZE, shuffle=True, drop_last=True,
                               collate_fn=chain_collate)
else:
    print("train_chain_records bellekte yok -- chain_loader SEQ2 hucresinde kurulacak.")


In [ ]:
def project_chain_batch_self_infer(model, batch: "ChainBatch"):
    """SelfInferringHead için zincir-düğümü projeksiyonu -- v4 ile aynı."""
    B, L, D = batch.emb.shape
    flat = batch.emb.view(B * L, D)
    mode_vec, _ = model.infer_mode(flat, flat)
    z_img = model.img_proj(flat, mode_vec)
    z_txt = model.txt_proj(flat, mode_vec)
    mod = batch.modality.view(B * L, 1).float()
    z = z_img * (1 - mod) + z_txt * mod
    return z.view(B, L, -1)


In [ ]:
# HÜCRE EVAL: leak-free degerlendirme (ana notebook ile birebir ayni)
@torch.no_grad()
def evaluate_self_paired(model, pairs):
    """D / D+SACL-lite icin dogru, sizintisiz degerlendirme -- egitimle TUTARLI
    (kendine-eslenmis mod cikarimi hem egitimde hem testte kullanilir)."""
    model.eval()
    img_all = torch.tensor(np.stack([np.asarray(p["img_embedding"], dtype=np.float32) for p in pairs])).to(device)
    txt_all = torch.tensor(np.stack([np.asarray(p["txt_embedding"], dtype=np.float32) for p in pairs])).to(device)
    mode_vec_img, _ = model.infer_mode(img_all, img_all)
    mode_vec_txt, _ = model.infer_mode(txt_all, txt_all)
    z_img = model.img_proj(img_all, mode_vec_img)
    z_txt = model.txt_proj(txt_all, mode_vec_txt)
    sim = (z_txt @ z_img.T).cpu().numpy()
    n = len(pairs)
    r1 = r5 = r10 = 0
    mrr = 0.0
    for i in range(n):
        order = np.argsort(-sim[i])
        rank = int(np.where(order == i)[0][0]) + 1
        if rank == 1: r1 += 1
        if rank <= 5: r5 += 1
        if rank <= 10: r10 += 1
        mrr += 1.0 / rank
    return {"R@1": r1/n*100, "R@5": r5/n*100, "R@10": r10/n*100, "MRR": mrr/n}


In [ ]:
# HÜCRE D_SACL_FOCUS1: D+SACL-lite egitim fonksiyonu (epoch/sure izlemeli)
import time

def train_d_sacl_lite_leakfree_timed(epochs=75, lambda_seq=0.1, pos_window=2, seed=None,
                                      val_pairs=None, patience=4, eval_every=5):
    """train_d_sacl_lite_leakfree ile AYNI egitim mantigi -- ek olarak kac
    epoch calistigini, en iyi val sonucunun hangi epoch'ta bulundugunu, ve
    toplam egitim suresini (saniye) de dondurur."""
    t0 = time.time()
    torch.manual_seed(seed if seed is not None else SEED)
    model = SelfInferringHead().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    chain_iter = itertools.cycle(chain_loader)
    best_state = copy.deepcopy(model.state_dict())
    best_val_r1 = -1.0
    best_epoch = 0
    checks_since_improve = 0
    epochs_run = 0

    for epoch in range(epochs):
        epochs_run = epoch + 1
        model.train()
        epoch_loss = epoch_aux = epoch_seq = 0.0
        for batch in train_loader:
            img = batch["img"].to(device); txt = batch["txt"].to(device)
            hn = batch["hn"].to(device); has_hn = batch["has_hn"].to(device)
            mode_idx = batch["mode_idx"].to(device)

            optimizer.zero_grad()
            mode_vec_img, mode_logits = model.infer_mode(img, img)
            mode_vec_txt, _ = model.infer_mode(txt, txt)
            z_img = model.img_proj(img, mode_vec_img)
            z_txt = model.txt_proj(txt, mode_vec_txt)
            z_hn = model.project_image_with_vec(hn, mode_vec_img) if has_hn.sum() > 0 else None

            base_loss = hn_info_nce(z_img, z_txt, z_hn=z_hn, has_hn=has_hn)
            mask = has_hn.bool()
            aux = F.cross_entropy(mode_logits[mask], mode_idx[mask]) if mask.sum() > 0 else torch.tensor(0.0, device=device)

            cb = next(chain_iter).to(device)
            z_chain = project_chain_batch_self_infer(model, cb)
            l_seq, _ = sequence_aware_loss(z_chain, cb, pos_window=pos_window,
                                            lambda_multipos=LAMBDA_MULTIPOS, lambda_rank=LAMBDA_RANK,
                                            margin=RANK_MARGIN, same_modality_only=True)

            loss = base_loss + MODE_AUX_WEIGHT * aux + lambda_seq * l_seq
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item(); epoch_aux += aux.item(); epoch_seq += l_seq.item()
        scheduler.step()
        if (epoch + 1) % 25 == 0 or epoch == 0:
            n = len(train_loader)
            print(f"  [D+SACL-lite-timed] epoch {epoch+1}/{epochs} loss={epoch_loss/n:.4f} lr={scheduler.get_last_lr()[0]:.2e}")

        if val_pairs is not None and ((epoch + 1) % eval_every == 0 or epoch + 1 == epochs):
            r_val = evaluate_self_paired(model, val_pairs)
            if r_val["R@1"] > best_val_r1:
                best_val_r1 = r_val["R@1"]
                best_epoch = epoch + 1
                best_state = copy.deepcopy(model.state_dict())
                checks_since_improve = 0
            else:
                checks_since_improve += 1
            if checks_since_improve >= patience:
                print(f"  [erken durdurma] epoch {epoch+1}'de durduruldu (en iyi val R@1=%{best_val_r1:.1f}, epoch {best_epoch}).")
                break

    elapsed = time.time() - t0
    if val_pairs is not None:
        model.load_state_dict(best_state)
    return model, {"epochs_run": epochs_run, "best_epoch": best_epoch,
                    "best_val_r1": best_val_r1, "elapsed_sec": elapsed}


## 9. 5-seed eğitim ve karşılaştırma

In [ ]:
# HÜCRE SEQ2: yeni sekans verisiyle 5-seed D+SACL kosusu
# DIKKAT: train_loader ve chain_loader bu hucrede YENI veriye baglanir.
# 600-sekans kosusuna donmek istersen HUCRE 12 ve HUCRE 15'i tekrar calistir.
train_loader = DataLoader(PairDataset(seq_train_pairs), batch_size=BATCH_SIZE,
                           shuffle=True, drop_last=True)
chain_loader = DataLoader(ChainDataset(seq_chain_records, max_len=SACL_MAX_LEN),
                           batch_size=CHAIN_BATCH_SIZE, shuffle=True, drop_last=True,
                           collate_fn=chain_collate)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
if "_ram" in globals(): _ram("loader kuruldu")
print(f"train_loader: {len(train_loader)} batch  |  chain_loader: {len(chain_loader)} batch")
assert len(train_loader) > 0 and len(chain_loader) > 0, "loader bos -- veri uretimi basarisiz"

SEQ_CKPT = f"{SEQ_RESULTS_DIR}/dsacl_focus_results.pkl"
if os.path.exists(SEQ_CKPT):
    with open(SEQ_CKPT, "rb") as f:
        seq_results = pickle.load(f)
    print(f"Onbellek: {len(seq_results)} seed tamam.")
else:
    seq_results = {}

for sd in [42, 123, 777, 2024, 31415]:
    if sd in seq_results:
        print(f"seed={sd} zaten var, atlaniyor."); continue
    print(f"\n=== D+SACL-lite, {NUM_SEQUENCES_BIG} sekans, seed={sd} ===")
    model, meta = train_d_sacl_lite_leakfree_timed(seed=sd, val_pairs=val_pairs)
    r = evaluate_self_paired(model, test_pairs)
    meta.update({"R@1": r["R@1"], "R@5": r["R@5"], "R@10": r["R@10"], "MRR": r["MRR"]})
    seq_results[sd] = meta
    print(f"  epoch={meta['epochs_run']} en_iyi={meta['best_epoch']} "
          f"sure={meta['elapsed_sec']:.1f}sn  R@1={meta['R@1']:.1f}%  MRR={meta['MRR']:.3f}")
    with open(SEQ_CKPT, "wb") as f:
        pickle.dump(seq_results, f)
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
print(f"\nTamam: {len(seq_results)}/5 seed -> {SEQ_CKPT}")


In [ ]:
# HÜCRE SEQ3: 600 vs NUM_SEQUENCES_BIG sekans karsilastirmasi
from scipy import stats as _st
_prev_ckpt = f"{globals().get('RESULTS_DIR', DRIVE_BASE + '/ZoomInOut/v5i_02_results')}/dsacl_focus_results.pkl"
assert os.path.exists(_prev_ckpt), f"600-sekans sonucu bulunamadi: {_prev_ckpt}"
with open(_prev_ckpt, "rb") as f: _prev = pickle.load(f)
with open(f"{SEQ_RESULTS_DIR}/dsacl_focus_results.pkl", "rb") as f: _new = pickle.load(f)

_seeds = sorted(set(_prev) & set(_new))
print(f"Ortak seed: {_seeds}\n")
def _col(d, k): return np.array([d[s][k] for s in _seeds], dtype=float)

print(f"{'metrik':<22} {'600 sekans':>18} {str(NUM_SEQUENCES_BIG)+' sekans':>18}")
print("-"*60)
for k, lab, f in [("R@1","R@1 (%)","{:.2f}"), ("MRR","MRR","{:.3f}"),
                  ("epochs_run","Ort. epoch","{:.1f}"), ("best_epoch","Ort. en-iyi epoch","{:.1f}"),
                  ("elapsed_sec","Ort. sure (dk)","{:.1f}")]:
    a, b = _col(_prev,k), _col(_new,k)
    if k == "elapsed_sec": a, b = a/60, b/60
    if k in ("R@1","MRR"):
        print(f"{lab:<22} {f.format(a.mean())+' ± '+f.format(a.std(ddof=1)):>18} "
              f"{f.format(b.mean())+' ± '+f.format(b.std(ddof=1)):>18}")
    else:
        print(f"{lab:<22} {f.format(a.mean()):>18} {f.format(b.mean()):>18}")

r1a, r1b = _col(_prev,"R@1"), _col(_new,"R@1")
t_p, p_p = _st.ttest_rel(r1b, r1a)
t_i, p_i = _st.ttest_ind(r1b, r1a, equal_var=False)
print(f"\nFark: {r1b.mean()-r1a.mean():+.2f} puan ({NUM_SEQUENCES_BIG} sekans lehine ise pozitif)")
print(f"  eslestirilmis t-test : t={t_p:.3f}  p={p_p:.4f}")
print(f"  bagimsiz t-test      : t={t_i:.3f}  p={p_i:.4f}")
with open(SEQ_TRAIN_PAIRS_PATH, "rb") as f: _cov_new = pickle.load(f).get("coverage_pct")
print(f"\nTrain kapsami: 600 sekans -> %43.7 (raporlanan)   |   {NUM_SEQUENCES_BIG} sekans -> "
      + (f"%{_cov_new:.1f}" if _cov_new else "onbellekte yok"))
with open(f"{SEQ_RESULTS_DIR}/comparison_summary.pkl", "wb") as f:
    pickle.dump({"prev": _prev, "new": _new, "coverage_new": _cov_new,
                 "paired": (float(t_p), float(p_p)), "indep": (float(t_i), float(p_i))}, f)
print("✅ Ozet kaydedildi.")


---
## Son hücre: önerilen yapılandırma (800 sekans)

Bu notebook şimdiye kadar yalnızca SACL hard window kolunu koştu. Makale
artık o yapılandırmayı önermediği için aynı koşulda zincir kaybı olmayan
kol da gerekiyor. Aşağıdaki hücre onu koşar.

**Önce yukarıdaki hücreleri sırayla çalıştır.** Sekans üretimi hücresinde
"önbellekten yükleniyor" görmelisin; "sıfırdan üretiliyor" yazarsa DURDUR.


In [ ]:
# HÜCRE PS-D: önerilen yapılandırma (zincir kaybı YOK) bu sekans koşulunda
# ---------------------------------------------------------------------------
# Bu notebook simdiye kadar YALNIZCA train_d_sacl_lite_leakfree_timed kolunu
# kosmustu, yani makaleden cikardigimiz SACL hard window yapilandirmasini.
# Olcek ve sekans tablolarinin onerilen yontemi raporlamasi icin ayni kosulda
# zincir kaybi OLMAYAN kol da gerekiyor.
#
# lambda_seq=0 ile cagirmak dogru sonucu verirdi ama zincir yigininin ileri ve
# geri gecisi yine yapilacagi icin hizlanma saglamazdi. Bu surum zincir
# hesabini TAMAMEN atlar; egitim yaklasik on kat hizlanir.
import os, time, copy, pickle
import numpy as np, torch, torch.nn.functional as F
from torch.utils.data import DataLoader

CKPT_D = f"{SEQ_RESULTS_DIR}/d_leakfree_results.pkl"
# Bu notebook'ta SEEDS tanimli degil; tohum listesi SACL hucresinde
# dogrudan yazilmis. Ayni bes tohum kullanilir.
SEEDS = globals().get("SEEDS", [42, 123, 777, 2024, 31415])

def train_D_leakfree_timed(epochs=75, seed=None, val_pairs=None, patience=4, eval_every=5):
    """train_d_sacl_lite_leakfree_timed ile AYNI egitim mantigi; tek fark
    zincir kaybinin hic hesaplanmamasi."""
    t0 = time.time()
    torch.manual_seed(seed if seed is not None else SEED)
    model = SelfInferringHead().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    # Bu kosulun ciftlerinden ACIKCA kendi yukleyicisini kurar -- global
    # train_loader hucre sirasina gore 600 ya da 800/3000'e bagli olabilir.
    dl = DataLoader(PairDataset(seq_train_pairs), batch_size=BATCH_SIZE,
                    shuffle=True, drop_last=True)
    best_state = copy.deepcopy(model.state_dict())
    best_val_r1 = -1.0; best_epoch = 0; checks_since_improve = 0; epochs_run = 0

    for epoch in range(epochs):
        epochs_run = epoch + 1
        model.train()
        for batch in dl:
            img = batch["img"].to(device); txt = batch["txt"].to(device)
            hn = batch["hn"].to(device); has_hn = batch["has_hn"].to(device)
            mode_idx = batch["mode_idx"].to(device)

            optimizer.zero_grad()
            mode_vec_img, mode_logits = model.infer_mode(img, img)
            mode_vec_txt, _ = model.infer_mode(txt, txt)
            z_img = model.img_proj(img, mode_vec_img)
            z_txt = model.txt_proj(txt, mode_vec_txt)
            z_hn = model.project_image_with_vec(hn, mode_vec_img) if has_hn.sum() > 0 else None

            loss = hn_info_nce(z_img, z_txt, z_hn=z_hn, has_hn=has_hn)
            mask = has_hn.bool()
            if mask.sum() > 0:
                loss = loss + MODE_AUX_WEIGHT * F.cross_entropy(mode_logits[mask], mode_idx[mask])
            loss.backward(); optimizer.step()
        scheduler.step()

        if val_pairs is not None and ((epoch + 1) % eval_every == 0 or epoch + 1 == epochs):
            r_val = evaluate_self_paired(model, val_pairs)
            if r_val["R@1"] > best_val_r1:
                best_val_r1 = r_val["R@1"]; best_epoch = epoch + 1; checks_since_improve = 0
                best_state = copy.deepcopy(model.state_dict())
            else:
                checks_since_improve += 1
                if checks_since_improve >= patience: break
    model.load_state_dict(best_state)
    return model, {"epochs_run": epochs_run, "best_epoch": best_epoch,
                   "best_val_r1": best_val_r1, "elapsed_sec": time.time() - t0}

SONUC_D = pickle.load(open(CKPT_D, "rb")) if os.path.exists(CKPT_D) else {}
print(f"sekans sayisi = {NUM_SEQUENCES_BIG} | cift sayisi = {len(seq_train_pairs)}")
print(f"cikti -> {CKPT_D}\n")
for sd in SEEDS:
    if sd in SONUC_D: print(f"  seed {sd} zaten var, atlaniyor."); continue
    m, meta = train_D_leakfree_timed(seed=sd, val_pairs=val_pairs)
    meta.update(evaluate_self_paired(m, test_pairs))
    SONUC_D[sd] = meta
    print(f"  seed {sd:<6} epoch={meta['epochs_run']:<3} en_iyi={meta['best_epoch']:<3} "
          f"{meta['elapsed_sec']/60:5.1f} dk   R@1={meta['R@1']:.1f}%  MRR={meta['MRR']:.3f}")
    del m
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    pickle.dump(SONUC_D, open(CKPT_D, "wb"))

r1 = np.array([SONUC_D[s]["R@1"] for s in SEEDS]); mr = np.array([SONUC_D[s]["MRR"] for s in SEEDS])
dk = np.mean([SONUC_D[s]["elapsed_sec"] for s in SEEDS]) / 60
print(f"\n{NUM_SEQUENCES_BIG} sekans, zincir kaybi YOK (onerilen yapilandirma):")
print(f"  R@1 = {r1.mean():.2f} ± {r1.std(ddof=1):.2f}")
print(f"  MRR = {mr.mean():.3f} ± {mr.std(ddof=1):.3f}")
print(f"  ort. egitim suresi = {dk:.1f} dk")
print(f"  uretilen cift = {len(seq_train_pairs)}")
